> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# Entrainment Heads → Agent Memory： complete  Colab notebook

 Its from the old book.  **`meta-llama/Llama-3.1-8B`（base）+ TransformerLens 2.17.0 +  Freeze Model  + Gumbel head-mask  Search **。
 Old book: [semantic_operator_family_generalization.ipynb](https://colab.research.google.com/drive/1uXs0B9yFWdk2rpq8mz5oy6dkSyuUWVzK)。

 Ben.  notebook  Data achieved, Actor  Tool execution, Writer  Free generation, reading. / Write.  heads  Search, check, check. M0/M1、2×2  downstream tasks, local value replacement, optional online updates, checkpoints, and statistics. All implementation is embedded; no separate upload is needed  Python  Documentation. 

** For the first time: **  Select to Load Old Books  8B  Model.  GPU， Run installation first  cell； If the loaded package is replaced, the session is restarted and the session is re-opened  imports  Execute down. Default  `QUICK_MODE=True`、 confirmation  test  Close. Login.  Hugging Face  and to ensure that the account number has access to the model; Drive  To save the interrupted recovery file. First look at the model first.  smoke  And capacity.  pilot  The output. 

** Three different levels: **  Fixed legal prefix list  token margin  For positioning.  heads； complete  JSON  Tools for evaluation  Actor； complete  JSON  Memory for evaluation  Writer  and downstream tasks. Do not treat  margin  Improve Direct Writing  agent  Improved success rate. 

 The distractor is a true but irrelevant city fact about another customer; correct evidence remains in context. The main experiment does not directly insert a false target value  memory。 Partial replacement is a separately identified mechanism diagnosis. 

** This edition is based on  handoff  Revised development version. **  Default only running  pilot（`RUN_DEVELOPMENT=False`）。8   pilot source  Overwrite Template  ×  Preferred inventory  × memory  All in the order  8  The grouping; this remains a small sample feasibility test. Actor  Use two independent, complete, executable demonstrations. The model still freely generates every action in the actual task; tools neither supply missing actions nor correct answers. 

v1/v2  Seen  3/6 clean  Success rate from user  Colab  trajectories; v4  Not yet  8B GPU  execute without assuming improved success. These software checks are not research findings. proposal  and realized  primary endpoint、 Capacity maintenance standards and official sample levels are still to be harmonized and formalized  test  Keep it closed. 

## 00 ·  Dependence 

 Keep the old book all the majors  imports。 To avoid the old version,  TransformerLens  Disrupted by new dependence, fixed.  `transformers`、`datasets`、`pyarrow`  e.g. compatible combinations. Python 3.13  Use  NumPy 2.1.3；Python 3.10–3.12  Use  NumPy 1.26.4。 Install with the first code cell once, restart the runtime after success, and continue from the second cell. Torch  Reuse an existing runtime satisfying  `>=2.6`   CUDA  Version. 

In [1]:
#  First use: Run this block and restart after installation  Colab  Run when. 
import sys
import subprocess

python_version = sys.version_info[:2]
print(f"Python: {sys.version.split()[0]}\n Interpreter : {sys.executable}", flush=True)
if not ((3, 10) <= python_version <= (3, 13)):
    raise RuntimeError(
        f" current  Python {sys.version.split()[0]}  Not applicable to Ben  notebook  A fixed dependency combination. "
        " Please switch to  Python 3.10–3.13   Colab  Try again when running. "
    )

numpy_version = "2.1.3" if python_version == (3, 13) else "1.26.4"
print(f" Select when running this one  NumPy {numpy_version}", flush=True)

packages = [
    "transformer_lens==2.17.0",
    "transformers==4.57.1",
    "huggingface_hub==0.36.0",
    f"numpy=={numpy_version}",
    "pandas==2.2.3",
    "plotly==6.1.2",
    "scipy==1.15.3",
    "scikit-learn==1.6.1",
    "datasets==3.6.0",
    "pyarrow==20.0.0",
    "tqdm",
]
print(" Start installation of dependency; maintain a full log to check. ", flush=True)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        " Reliance on installation failed. Please look above  pip  First in the log  ERROR，"
        " Reruns the block after repair; for the time being, do not run the subsequent block. "
    ) from exc

print("\n Install command successfully executed. Please be on  Colab  . When running again in the menu, "
      " Continue from the second code cell without rerunning installation. ", flush=True)

Python: 3.13.16
 Interpreter : /usr/bin/python3
 Select when running this one  NumPy 2.1.3
 Start installation of dependency; maintain a full log to check. 

 Install command successfully executed. Please be on  Colab  restart the runtime from the menu, then continue from the second code cell without rerunning installation. 


In [2]:
#  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
import sys
import importlib.metadata as metadata
import importlib.util

print("Python:", sys.version.split()[0])
print(" Interpreter :", sys.executable)
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

expected_versions = {
    "transformer-lens": "2.17.0", "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
    "pandas": "2.2.3",
    "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
    "datasets": "3.6.0", "pyarrow": "20.0.0",
}
problems = []
for package, expected in expected_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
    if installed != expected:
        problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

for package in ("torch", "tqdm"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        problems.append(f"{package}:  not installed ")

try:
    torch_version = metadata.version("torch")
    torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
    if torch_major_minor < (2, 6):
        problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
except metadata.PackageNotFoundError:
    pass  #  The record is missing. 
except ValueError:
    problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

if importlib.util.find_spec("transformer_lens") is None:
    problems.append(" current  Python  Cannot find  transformer_lens  module ")
if problems:
    raise RuntimeError(
        " Reliance check failed: \n- " + "\n- ".join(problems)
        + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
          " Go on from this block. "
    )
print(" Reliance on a version check pass; continues to run the third code block. ")

Python: 3.13.16
 Interpreter : /usr/bin/python3
transformer-lens: 2.17.0 ( Yes.  2.17.0)
transformers: 4.57.1 ( Yes.  4.57.1)
huggingface-hub: 0.36.0 ( Yes.  0.36.0)
numpy: 2.1.3 ( Yes.  2.1.3)
pandas: 2.2.3 ( Yes.  2.2.3)
plotly: 6.1.2 ( Yes.  6.1.2)
scipy: 1.15.3 ( Yes.  1.15.3)
scikit-learn: 1.6.1 ( Yes.  1.6.1)
datasets: 3.6.0 ( Yes.  3.6.0)
pyarrow: 20.0.0 ( Yes.  20.0.0)
torch: 2.11.0+cu130
tqdm: 4.67.3
 Reliance on a version check pass; continues to run the third code block. 


In [3]:
#  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
import gc, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, utils
from huggingface_hub import notebook_login, get_token, model_info
import importlib.metadata as metadata
from IPython.display import display

PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
PACKAGE_VERSIONS = {name: metadata.version(name) for name in PACKAGE_NAMES}
print(PACKAGE_VERSIONS)
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print(" Reliance on import success. ")

{'torch': '2.11.0+cu130', 'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0', 'numpy': '2.1.3', 'pandas': '2.2.3', 'datasets': '3.6.0', 'pyarrow': '20.0.0', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'plotly': '6.1.2'}
Python: 3.13.16
CUDA available: True
 Reliance on import success. 


## 01 ·  Configure and Save Directory 

QUICK  validates software and preliminary signals; intervals from small samples are not paper conclusions. FULL  Keep the old one.  K  candidate  `20/40/60/80`， Multiple Searches  seed  Report all conditions and freeze the first as primary beforehand; never select based on  test  Pick up.  seed。

 Change model, accuracy, measure of data, prompt、 After searching parameters or relying on them, use new  `RUN_NAME`。 Use the original directory when reconnecting with configurations, which restores case-by-case results and  mask  Optimizer /RNG。 Keeps the complete original output; failure cases are not removed from the denominator. 

In [4]:
QUICK_MODE = True
MODEL_NAME = "meta-llama/Llama-3.1-8B"  #  Same as old book: base， No, it is not.  Instruct
MODEL_REVISION = None  #  first parsing and locking  commit SHA； Continue reading records in the same directory 
REQUESTED_DTYPE = "bfloat16"  #  Same as the old book; no support is given without a clear statement of error and no secret cutting of precision 
MOUNT_DRIVE = True
RUN_NAME = "agent_entrainment_v4_reviewed_quick" if QUICK_MODE else "agent_entrainment_v4_reviewed_full"
RUN_DEVELOPMENT = False  #  Run first. 06 Inspection pilot； Confirmed as feasible True， From 07 Go on. 
RUN_ONLINE = False             #  Additional expenses; exploratory analysis independent of  2×2
FREEZE_AND_RUN_TEST = False    # FULL  Freezing protocol only works when the development is complete 

CFG = {
    "model_name": MODEL_NAME, "dtype": REQUESTED_DTYPE, "quick_mode": QUICK_MODE,
    "actor_prompt_version": "complete_demos_v4", "data_design": "crossed_2x2x2_v4",
    "data_seed": 1729, "n_pilot": 8 if QUICK_MODE else 24,
    "n_train": 16 if QUICK_MODE else 120, "n_dev": 8 if QUICK_MODE else 40,
    "n_test": 24 if QUICK_MODE else 120,
    "search_seeds": [11] if QUICK_MODE else [11, 23, 37],
    "random_seeds": [101] if QUICK_MODE else [101, 202, 303],
    "search_steps": 40 if QUICK_MODE else 450,
    "batch_size": 1, "lr": 0.08, "sparsity": 0.025,
    "temp_start": 1.0, "temp_end": 0.25, "clean_weight": 0.1,
    "top_k": [8, 16, 32] if QUICK_MODE else [20, 40, 60, 80],
    "max_clean_margin_drop": 0.25, "save_every": 5,
    "hook_kind": "z", "use_cache": True, "context_window": 4096, "max_prompt_tokens": 3072,
    "actor_max_new_tokens": 128, "writer_max_new_tokens": 384,
    "max_actions": 6, "followup_tasks": 3, "online_horizons": [1, 3, 5],
    "min_pilot_success": 0.75, "max_pilot_format_failure": 0.20,
    "min_natural_writer_errors": 2, "min_writer_valid": 0.80,
    "bootstrap_reps": 2000,
    "city_candidates": ["Paris", "Tokyo", "Rome", "London", "Berlin", "Boston", "Chicago", "Madrid", "Moscow", "Seattle", "Toronto", "Vienna", "Dublin", "Sydney", "Austin", "Denver"],
}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/AAAI_Workshop/agent_entrainment")
else:
    OUTPUT_ROOT = Path("/content/agent_entrainment")
    print(" currently saved on temporary run; download the output directory before the end. ")
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", RUN_DIR)
print("Sizes:", {k:v for k,v in CFG.items() if k.startswith("n_")})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs: /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v4_reviewed_quick
Sizes: {'n_pilot': 8, 'n_train': 16, 'n_dev': 8, 'n_test': 24}


## 02 ·  Core achievement 

 Next definition.  cells  You can fold. Default is in  `hook_z`  Up to zero.  query-head  Output, Reduce Old Books  `hook_result`  Middle billing; back.  smoke  Checks the consistency of the two interfaces. Search only updates  mask， Model parameters are always frozen. 

 All  forward  Im using a single line.  padding  Enter, Candidature  token  Must be satisfied.  `encode(prefix + value) == encode(prefix) + [one_token]`。 Free Generate Every New  KV cache， Intervention coverage  prefill  And every step.  decode。

In [5]:
# Core infrastructure: exact-prefix token probes, full generation, and search.
import contextlib
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import time
import warnings
import numpy as np
import torch
import torch.nn.functional as F


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def checkpoint_item(stage, key, payload, fn):
    """Only reuse an item produced by exactly this code/model/config/payload."""
    fingerprint = stable_hash({"implementation": IMPLEMENTATION_HASH,
                               "run": RUN_FINGERPRINT, "payload": payload})
    safe_key = re.sub(r"[^A-Za-z0-9_.-]", "_", str(key))[:150]
    path = RUN_DIR / "items" / stage / (safe_key + ".json")
    if path.exists():
        record = json.loads(path.read_text())
        if record["fingerprint"] != fingerprint:
            raise RuntimeError(f"Stale checkpoint: {path}. Use a new RUN_NAME; do not mix experiments.")
        return record["result"]
    result = fn()
    atomic_json(path, {"fingerprint": fingerprint, "payload_hash": stable_hash(payload), "result": result})
    return result


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result


def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks


class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}


def eligible_snapshots(backend, cases, role, condition, histories=None):
    retained, excluded = [], []
    for case in cases:
        history = None if histories is None else histories[case["source_id"]]
        snapshot = actor_snapshot(case, condition) if role == "actor" else writer_snapshot(case, condition, history)
        gold = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            excluded.append({"source_id": case["source_id"], "role": role, "condition": condition,
                             "gold_token": gold, "distractor_token": wrong})
        else:
            retained.append(snapshot)
    return retained, excluded


def snapshot_pairs(backend, cases, role, histories_by_condition=None):
    attacks, controls, exclusions = [], [], []
    for case in cases:
        pair = []
        for condition in ("distractor", "correct_only"):
            histories = None if histories_by_condition is None else histories_by_condition[condition]
            kept, dropped = eligible_snapshots(backend, [case], role, condition, histories)
            pair.append(kept)
            exclusions.extend(dropped)
        if all(pair):
            attacks.append(pair[0][0]); controls.append(pair[1][0])
    if not attacks:
        raise RuntimeError("No eligible diagnostic pairs. Inspect tokenizer prefixes and report exclusions; do not drop behavioral tasks.")
    return attacks, controls, exclusions


def gumbel_drop(logits, temperature):
    uniform = torch.rand_like(logits).clamp(1e-6, 1 - 1e-6)
    soft = torch.sigmoid((logits + uniform.log() - torch.log1p(-uniform)) / temperature)
    hard = (soft > 0.5).to(soft.dtype)
    return hard.detach() - soft.detach() + soft


def _save_search(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(record, temporary)
    os.replace(temporary, path)


def train_mask(backend, attacks, controls, cfg, name, seed):
    """Optimize only mask logits. Test episodes never enter this function."""
    if any(s["split"] != "train" for s in attacks + controls):
        raise ValueError("Head search accepts training snapshots only")
    if [s["source_id"] for s in attacks] != [s["source_id"] for s in controls]:
        raise ValueError("Training attack/control pairs are not aligned")
    model = backend.model
    shape = (model.cfg.n_layers, model.cfg.n_heads)
    signature = stable_hash({"run": RUN_FINGERPRINT, "code": IMPLEMENTATION_HASH,
                             "attacks": attacks, "controls": controls, "cfg": cfg, "seed": seed, "name": name})
    path = RUN_DIR / "search" / f"{name}-seed{seed}.pt"
    seed_everything(seed)
    with torch.enable_grad():
        logits = torch.nn.Parameter(torch.full(shape, -4.0, device=backend.device, dtype=torch.float32))
        optimizer = torch.optim.AdamW([logits], lr=cfg["lr"], weight_decay=0.0)
        start, history = 0, []
        if path.exists():
            # Only load checkpoints produced by this notebook in your own run directory.
            state = torch.load(path, map_location=backend.device, weights_only=False)
            if state["signature"] != signature:
                raise RuntimeError("Search checkpoint mismatch. Use a new RUN_NAME.")
            logits.data.copy_(state["logits"])
            optimizer.load_state_dict(state["optimizer"])
            start, history = state["step"], state["history"]
            random.setstate(state["python_rng"])
            np.random.set_state(state["numpy_rng"])
            torch.set_rng_state(state["torch_rng"].cpu())
            if torch.cuda.is_available() and state["cuda_rng"] is not None:
                torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda_rng"]])
        with torch.no_grad():
            clean_reference = [float(backend.margin(s).item()) for s in controls]
        for step in range(start, cfg["search_steps"]):
            ratio = step / max(1, cfg["search_steps"] - 1)
            temperature = cfg["temp_start"] * (cfg["temp_end"] / cfg["temp_start"]) ** ratio
            indices = random.sample(range(len(attacks)), min(cfg["batch_size"], len(attacks)))
            optimizer.zero_grad(set_to_none=True)
            total, attack_value, clean_penalty = 0.0, 0.0, 0.0
            # Separate backward per example reduces retained activation memory.
            for index in indices:
                drop = gumbel_drop(logits, temperature)
                attacked_margin = backend.margin(attacks[index], drop_mask=drop)
                clean_margin = backend.margin(controls[index], drop_mask=drop)
                harm = F.relu(torch.tensor(clean_reference[index], device=backend.device) - clean_margin)
                loss = (-attacked_margin + cfg["clean_weight"] * harm + cfg["sparsity"] * drop.sum()) / len(indices)
                loss.backward()
                total += float(loss.detach()); attack_value += float(attacked_margin.detach()) / len(indices)
                clean_penalty += float(harm.detach()) / len(indices)
            if logits.grad is None or not torch.isfinite(logits.grad).all():
                raise RuntimeError("Invalid mask gradient; search did not run correctly")
            grad_norm = float(logits.grad.norm())
            torch.nn.utils.clip_grad_norm_([logits], 5.0)
            optimizer.step()
            history.append({"step": step + 1, "loss": total, "attack_margin": attack_value,
                            "clean_harm": clean_penalty, "temperature": temperature, "grad_norm": grad_norm,
                            "expected_heads": float(torch.sigmoid(logits.detach()).sum())})
            if (step + 1) % cfg["save_every"] == 0 or step + 1 == cfg["search_steps"]:
                _save_search(path, {"signature": signature, "step": step + 1,
                                   "logits": logits.detach(), "optimizer": optimizer.state_dict(), "history": history,
                                   "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                                   "torch_rng": torch.get_rng_state(),
                                   "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})
                print(f"{name} seed={seed}: {step+1}/{cfg['search_steps']}, margin={attack_value:.3f}, grad={grad_norm:.3g}")
        ranking = torch.argsort(logits.detach().flatten(), descending=True).cpu().tolist()
        heads = [[i // shape[1], i % shape[1]] for i in ranking]
        result = {"name": name, "seed": seed, "ranked_heads": heads,
                  "scores": logits.detach().cpu().tolist(), "history": history, "signature": signature}
    assert all(not p.requires_grad for p in model.parameters())
    atomic_json(RUN_DIR / "search" / f"{name}-seed{seed}.json", result)
    return result


def select_heads_dev(backend, ranking, attacks, controls, cfg, name):
    if any(s["split"] != "dev" for s in attacks + controls):
        raise ValueError("K selection accepts development snapshots only")
    if not attacks or len(attacks) != len(controls):
        raise ValueError("Need aligned nonempty development snapshots")
    if [s["source_id"] for s in attacks] != [s["source_id"] for s in controls]:
        raise ValueError("Development attack/control source IDs are not aligned")
    rows = []
    with torch.no_grad():
        base_attack = np.mean([float(backend.margin(s)) for s in attacks])
        base_clean = np.mean([float(backend.margin(s)) for s in controls])
        for k in cfg["top_k"]:
            heads = ranking["ranked_heads"][:k]
            payload = {"heads": heads, "attacks": attacks, "controls": controls, "cfg": cfg}
            def evaluate():
                return {"k": k, "heads": heads,
                        "attack_margin": float(np.mean([float(backend.margin(s, heads)) for s in attacks])),
                        "clean_margin": float(np.mean([float(backend.margin(s, heads)) for s in controls]))}
            row = checkpoint_item("dev_selection", f"{name}-seed{ranking['seed']}-k{k}", payload, evaluate)
            row = dict(row, attack_gain=row["attack_margin"] - float(base_attack),
                       clean_change=row["clean_margin"] - float(base_clean))
            layer_counts = {}
            for layer, _ in heads:
                layer_counts[layer] = layer_counts.get(layer, 0) + 1
            row["disjoint_random_control_possible"] = all(count * 2 <= backend.model.cfg.n_heads for count in layer_counts.values())
            row["eligible"] = (row["attack_gain"] > 0 and row["clean_change"] >= -cfg["max_clean_margin_drop"]
                               and row["disjoint_random_control_possible"])
            rows.append(row)
    eligible = [r for r in rows if r["eligible"]]
    chosen = max(eligible, key=lambda r: (r["attack_gain"], -r["k"])) if eligible else None
    result = {"name": name, "seed": ranking["seed"], "candidates": rows,
              "base_attack_margin": float(base_attack), "base_clean_margin": float(base_clean),
              "selected": chosen, "heads": chosen["heads"] if chosen else [], "eligible": chosen is not None}
    atomic_json(RUN_DIR / "search" / f"{name}-seed{ranking['seed']}-dev.json", result)
    return result

###  Taskable environment 

 Each task has a real inventory, booking number and order status. Actor  Use tool feedback to choose the main or fallback item. The environment does not know the correct city and does not correct wrong orders. `task_success`  The results of the evaluation of the orders, `stock_check_adherence`  Check separately whether inventory is checked first. 

train/dev/test  Clients.  ID、 Separate entity-city bindings and task-template families; the city vocabulary is shared, so this is not  held-out vocabulary  Test. Every one.  source  The jamming variant is always the same as the follow-up mission.  split。 Every 8 source Cross-covering of two templates, two preferred inventories and two memory order;inadequate 8 The end blocks are not completely balanced. `matched_neutral`  To Approximate Length  metadata  By comparison, actual  token  Numbers will be saved. 

In [6]:
# Deterministic task data, strict JSON protocol, and executable Actor environment.
# This section requires only the Python standard library. The model backend is
# defined earlier in the notebook; no model weights are changed by this section.
import copy
import json
import random
import re
import time


TASK_TEMPLATES = {
    "pilot": [
        ("pilot_order", "Create one order for {target}. Try {preferred} first; use {backup} only if the preferred item is unavailable."),
        ("pilot_dispatch", "Arrange a delivery for {target}: preferred item {preferred}, fallback item {backup}. Check availability before reserving."),
    ],
    "train": [
        ("train_purchase", "Complete a purchase for customer {target}. Their first-choice item is {preferred}; the permitted substitute is {backup}."),
        ("train_fulfillment", "Fulfill {target}'s request for {preferred}. If stock prevents that choice, fulfill it with {backup}."),
    ],
    "dev": [
        ("dev_request", "Customer {target} needs an order. Select {preferred} when available and otherwise select {backup}."),
        ("dev_shipment", "Prepare one shipment to {target}. Item priority: {preferred} before {backup}, with substitution allowed only when stock is unavailable."),
    ],
    "test": [
        ("test_checkout", "Finish checkout on behalf of {target}, preferring {preferred}. The only acceptable out-of-stock replacement is {backup}."),
        ("test_procurement", "Procure an item for {target}: obtain {preferred} if possible, else obtain {backup}. Place exactly one resulting order."),
    ],
}

CONDITIONS = (
    "correct_only", "distractor", "matched_neutral",
    "distractor_value_swap", "useful_memory", "legitimate_update",
)


def build_cases(split, n, seed, city_values):
    """Generate independent source tasks; all condition variants share a source_id.

    Target values are randomized per source. City labels may be shared across
    splits, but customer IDs and two template families per split are held out.
    This is template-family and entity generalization, not held-out vocabulary.
    Token validity must additionally be checked at each actual model prefix.
    """
    if split not in TASK_TEMPLATES:
        raise ValueError("split must be pilot, train, dev, or test")
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    values = list(city_values)
    if len(values) < 4 or len(set(values)) != len(values):
        raise ValueError("city_values requires >=4 distinct strings")
    if any(not isinstance(v, str) or not v or v.strip() != v or
           any(ord(c) < 32 for c in v) for v in values):
        raise ValueError("city_values must be nonempty printable, unpadded strings")
    rng = random.Random(f"{int(seed)}:{split}:order-task-v1")
    cases = []
    for index in range(n):
        target_city, other_city, neutral_city, updated_city = rng.sample(values, 4)
        # One complete 2 x 2 x 2 block per eight sources: template, stock,
        # and memory order vary independently. Partial blocks are incomplete.
        template_id, template = TASK_TEMPLATES[split][(index // 2) % 2]
        source_id = f"{split}-{int(seed)}-{index:05d}"
        preferred_item = f"item-{index % 7:02d}-primary"
        backup_item = f"item-{index % 7:02d}-backup"
        cases.append({
            "source_id": source_id, "split": split,
            "template_id": template_id, "task_template": template,
            "target_name": f"customer-{source_id}-B",
            "other_name": f"customer-{source_id}-A",
            "target_city": target_city, "distractor_city": other_city,
            "neutral_city": neutral_city, "updated_city": updated_city,
            "preferred_item": preferred_item, "backup_item": backup_item,
            "initial_env": {
                "stock": {preferred_item: int(index % 2 == 0), backup_item: 2},
                "reservations": {}, "orders": [],
                "next_reservation": 1, "next_order": 1,
            },
            "position_order": "target_first" if (index // 4) % 2 == 0 else "other_first",
        })
    return cases


def _preference_record(case, entity, value, suffix):
    return {"kind": "preference", "entity": entity,
            "attribute": "delivery_city", "value": value,
            "source_event": f"{case['source_id']}:{suffix}"}


def expected_city(case, condition):
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    return case["updated_city"] if condition == "legitimate_update" else case["target_city"]


def distractor_city(case, condition):
    # In the update control, the superseded value is the relevant wrong answer.
    if condition == "legitimate_update":
        return case["target_city"]
    if condition == "distractor_value_swap":
        return case["neutral_city"]
    return case["distractor_city"]


def memory_for_condition(case, condition):
    """Return observable records only; no evaluator labels enter model prompts.

    matched_neutral is *approximately* length-matched metadata, not a claimed
    token-exact control. useful_memory adds a true, relevant confirmation.
    no_memory is an optional dependence diagnostic, not an attack condition.
    """
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    if condition == "no_memory":
        return []
    target = _preference_record(case, case["target_name"], case["target_city"], "target:0")
    if condition == "correct_only":
        return [target]
    if condition == "legitimate_update":
        return [target, _preference_record(case, case["target_name"], case["updated_city"], "target:1")]
    if condition == "useful_memory":
        return [target, _preference_record(case, case["target_name"], case["target_city"], "target:confirmation")]
    if condition == "matched_neutral":
        extra = {"kind": "metadata", "entity": case["other_name"],
                 "attribute": "record_status", "value": "archived",
                 "source_event": f"{case['source_id']}:metadata:0"}
    else:
        extra = _preference_record(case, case["other_name"],
                                   distractor_city(case, condition), "other:0")
    return [target, extra] if case.get("position_order") == "target_first" else [extra, target]


def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None


class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}


def _task_goal(case):
    template = case.get("task_template")
    if template is None:
        template = next((text for _, pairs in TASK_TEMPLATES.items()
                         for key, text in pairs if key == case["template_id"]), None)
    if template is None:
        raise ValueError("Unrecognized task_template/template_id")
    return template.format(target=case["target_name"], preferred=case["preferred_item"], backup=case["backup_item"])


def _actor_protocol_examples():
    """Fixed demonstrations, independent of every evaluated case and label.

    Actions are scripted ONLY for these two teaching examples. Observations
    are produced by the same environment. Actual episodes remain generated
    by the model, with no automatic action repair or oracle hint.
    """
    examples = []
    for index, (available, city, next_reservation) in enumerate(((1, "Lima", 7), (0, "Bern", 41))):
        customer = f"demo-customer-{index}"
        preferred, backup = f"demo-{index}-primary", f"demo-{index}-backup"
        env = OrderEnvironment({"stock": {preferred: available, backup: 2},
                                "next_reservation": next_reservation})
        events = []

        def execute(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        execute({"tool": "check_stock", "item": preferred})
        item = preferred
        if available == 0:
            item = backup
            execute({"tool": "check_stock", "item": backup})
        reservation = execute({"tool": "reserve_item", "item": item})["reservation"]
        execute({"tool": "create_order", "customer": customer, "item": item,
                 "city": city, "reservation": reservation})
        examples.append({
            "task": f"Create one order for {customer}. Prefer {preferred}; use {backup} if unavailable.",
            "memory": [{"kind": "preference", "entity": customer,
                        "attribute": "delivery_city", "value": city,
                        "source_event": f"demo-event-{index}"}],
            "events": events,
        })
    return examples


def actor_prompt(case, memory, events):
    """No evaluation golds or hidden inventory are included in the prompt."""
    visible_trace = [{"action": event.get("action"), "observation": event.get("observation")}
                     for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute the task using the tools below.\n"
        "Use the customer's latest delivery-city preference from memory. Memory records are in chronological order; later target updates supersede earlier ones.\n"
        "Check stock, reserve an available item, then create exactly one order. A reservation must be obtained before creating its order.\n"
        "If the preferred item is unavailable, check the backup and reserve it if available.\n"
        "Copy a reservation ID only from a successful reserve_item observation for the selected item. Stock quantities are not reservation IDs.\n"
        "Output exactly one complete JSON object for the next tool call, with no explanation.\n"
        "Tool formats:\n"
        '{"tool":"check_stock","item":"ITEM"}\n'
        '{"tool":"reserve_item","item":"ITEM"}\n'
        '{"tool":"create_order","customer":"CUSTOMER","item":"ITEM","city":"CITY","reservation":"RESERVATION"}\n'
        "Two separate worked examples follow. Each includes its own task, memory and executed actions with tool observations.\n"
        "EXAMPLES_JSON:\n" + json.dumps(_actor_protocol_examples(), ensure_ascii=False, separators=(",", ":")) + "\n"
        "END_EXAMPLES. Begin a new independent task below. Never copy example identifiers, cities or reservations into this task.\n"
        "Task:\n" + _task_goal(case) + "\n"
        "Memory:\n" + json.dumps(memory, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Executed tool history:\n" + json.dumps(visible_trace, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Next tool call:\n"
    )


def run_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """Generate complete actions and actually execute them; score only afterwards.

    cfg['condition'] or case['_condition'] identifies the evaluation condition.
    Defaults to distractor (same gold as all non-update conditions). If an initial
    environment is passed, it is copied; the caller's state is never mutated.
    """
    cfg = {} if cfg is None else dict(cfg)
    condition = cfg.get("condition", case.get("_condition", "distractor"))
    gold = expected_city(case, condition)
    wrong = distractor_city(case, condition)
    initial = copy.deepcopy(case["initial_env"] if initial_env is None else initial_env)
    if isinstance(initial, OrderEnvironment):
        initial = initial.snapshot()
    env = OrderEnvironment(initial)
    max_actions = int(cfg.get("max_actions", 6))
    if max_actions < 1:
        raise ValueError("max_actions must be positive")
    events, actions = [], []
    token_count = prompt_tokens = model_calls = 0
    model_seconds = 0.0
    format_failure = False
    protocol_failure = False
    distractor_action = False
    checked_items = set()
    stock_check_adherence = True
    reservation_attempts = 0
    terminal = "action_budget"
    started = time.perf_counter()
    for step_index in range(max_actions):
        prompt = actor_prompt(case, memory, events)
        output = backend.generate(
            prompt, role="actor", heads=tuple(heads),
            max_new_tokens=cfg.get("actor_max_new_tokens", 128),
            use_cache=cfg.get("use_cache", True),
        )
        model_calls += 1
        raw_output = output.get("text", "")
        token_count += int(output.get("token_count", 0))
        prompt_tokens += int(output.get("prompt_tokens", 0))
        model_seconds += float(output.get("seconds", 0.0))
        action, parse_error = parse_json_output(raw_output, dict)
        event = {"step": step_index, "prompt": prompt, "action": action, "observation": None,
                 "raw_output": raw_output, "parse_error": parse_error,
                 "stop_reason": output.get("stop_reason")}
        if parse_error is not None:
            format_failure = True
            events.append(event)
            terminal = "format_failure"
            break
        actions.append(copy.deepcopy(action))
        if action.get("tool") == "reserve_item":
            reservation_attempts += 1
            stock_check_adherence = stock_check_adherence and isinstance(action.get("item"), str) and action.get("item") in checked_items and case["preferred_item"] in checked_items
        if action.get("tool") == "create_order" and action.get("city") == wrong:
            distractor_action = True
        observation = env.step(action)
        if action.get("tool") == "check_stock" and observation.get("ok"):
            checked_items.add(action["item"])
        event["observation"] = observation
        events.append(event)
        if observation.get("error") in {"action_must_be_object", "unknown_tool", "invalid_arguments"}:
            protocol_failure = True
            format_failure = True
            terminal = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            terminal = "order_created"
            break
    new_orders = env.state["orders"][len(initial.get("orders", [])):]
    preferred_available = initial["stock"].get(case["preferred_item"], 0) > 0
    gold_item = case["preferred_item"] if preferred_available else case["backup_item"]
    success = bool(len(new_orders) == 1 and new_orders[0]["customer"] == case["target_name"]
                   and new_orders[0]["item"] == gold_item and new_orders[0]["city"] == gold)
    result = {
        "source_id": case["source_id"], "split": case["split"], "condition": condition,
        "template_id": case["template_id"], "preferred_available": preferred_available,
        "position_order": case["position_order"],
        "success": success, "task_success": success,
        "distractor_action": distractor_action,
        "format_failure": format_failure, "protocol_failure": protocol_failure,
        "stock_check_adherence": bool(stock_check_adherence and reservation_attempts > 0),
        "budget_exhausted": terminal == "action_budget", "terminal": terminal,
        "actions": actions, "events": events, "final_state": env.snapshot(),
        "initial_state": initial, "token_count": token_count, "prompt_tokens": prompt_tokens,
        "seconds": time.perf_counter() - started, "model_seconds": model_seconds,
        "model_calls": model_calls, "action_count": len(actions),
        "accepted_order_count": len(new_orders),
        "gold_city": gold, "distractor_city": wrong, "gold_item": gold_item,
        "memory": copy.deepcopy(memory),
    }
    result["metrics"] = {k: result[k] for k in (
        "task_success", "distractor_action", "format_failure", "protocol_failure",
        "stock_check_adherence", "budget_exhausted", "action_count", "accepted_order_count")}
    return result


def frozen_history(case, condition, actor_result=None):
    """Freeze true preference evidence and, optionally, observed execution.

    A wrong action remains an observed action. It is never converted into a
    customer preference, and trajectories are never invented when omitted.
    """
    history = []
    for record in memory_for_condition(case, condition):
        event = copy.deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    if actor_result is not None:
        for event in actor_result["events"]:
            step = event["step"]
            if event.get("action") is not None:
                history.append({"kind": "observed_action", "sequence": len(history),
                                "source_event": f"{case['source_id']}:action:{step}",
                                "action": copy.deepcopy(event["action"])})
            else:
                history.append({"kind": "observed_output", "sequence": len(history),
                                "source_event": f"{case['source_id']}:output:{step}",
                                "text": event.get("raw_output", ""),
                                "parse_error": event.get("parse_error")})
            if event.get("observation") is not None:
                history.append({"kind": "tool_observation", "sequence": len(history),
                                "source_event": f"{case['source_id']}:observation:{step}",
                                "observation": copy.deepcopy(event["observation"])})
    return history


def actor_snapshot(case, condition):
    """An explicitly constructed offline probe, not a sampled Actor trajectory.

    The same valid inventory/reservation context is used for every memory
    condition of a source case. Teacher-forcing stops before the city value;
    full free-generation tests use run_actor and never receive this prefix.
    """
    env = OrderEnvironment(case["initial_env"])
    events = []
    preferred = case["preferred_item"]
    check = {"tool": "check_stock", "item": preferred}
    observation = env.step(check)
    events.append({"action": check, "observation": observation})
    item = preferred
    if observation["available"] <= 0:
        item = case["backup_item"]
        check = {"tool": "check_stock", "item": item}
        events.append({"action": check, "observation": env.step(check)})
    reserve = {"tool": "reserve_item", "item": item}
    reservation_observation = env.step(reserve)
    if not reservation_observation.get("ok"):
        raise ValueError("Snapshot requires at least one available allowed item")
    events.append({"action": reserve, "observation": reservation_observation})
    prefix_fields = {"tool": "create_order", "customer": case["target_name"],
                     "item": item, "reservation": reservation_observation["reservation"]}
    action_prefix = json.dumps(prefix_fields, ensure_ascii=False, separators=(",", ":"))[:-1] + ',"city":"'
    prompt = actor_prompt(case, memory_for_condition(case, condition), events) + action_prefix
    return {"source_id": case["source_id"], "split": case["split"],
            "role": "actor", "prompt": prompt,
            "gold": expected_city(case, condition),
            "distractor": distractor_city(case, condition), "condition": condition,
            "template_id": case["template_id"], "action_prefix": action_prefix,
            "probe_state": env.snapshot(), "probe_events": events,
            "snapshot_origin": "constructed_legal_prefix"}

### Writer  and  memory  Dissemination 

Writer  From frozen real experience  preference  memory. An incorrect order is an observed action, not automatically an incorrect preference. First measure natural misbinding; if the intact model  Writer  If too few natural errors occur, skip the writer branch automatically rather than manually contaminating the main experiment  memory  Manufacturing results. 

M0/M1  Share the same experience. 2×2   Ywr  First and foremost,  Writer  Intervention, second point.  Actor  Intervention; three follow-up tasks per grid re-set environment, memory  Read-only. Local replacements only change targets.  `value`， Retain the order of entries and sources and report separately the applicable ratio. 

In [7]:
# %% Writer: complete JSON memory generation and evidence-based scoring
import copy


def _latest_preference_records(history):
    """Latest explicit preference evidence only; actions/metadata are not facts."""
    if not isinstance(history, list):
        raise TypeError("history must be a list of observed event dictionaries")
    ordered = []
    for index, event in enumerate(history):
        if not isinstance(event, dict) or event.get("kind") != "preference":
            continue
        required = ("entity", "attribute", "value", "source_event")
        if any(not isinstance(event.get(k), str) for k in required):
            raise ValueError("Malformed preference evidence in frozen history")
        sequence = event.get("sequence", index)
        if not isinstance(sequence, (int, float)):
            raise ValueError("Preference sequence must be numeric")
        ordered.append((sequence, index, event))
    latest = {}
    for _, _, event in sorted(ordered, key=lambda x: (x[0], x[1])):
        key = (event["entity"], event["attribute"])
        latest[key] = {k: event[k] for k in
                       ("kind", "entity", "attribute", "value", "source_event")}
    return latest


def _memory_schema_error(memory):
    if not isinstance(memory, list):
        return "memory must be a JSON list"
    for index, record in enumerate(memory):
        if not isinstance(record, dict):
            return f"record {index} is not an object"
        required_keys = {"kind", "entity", "attribute", "value", "source_event"}
        if set(record) != required_keys:
            return f"record {index}: keys must be exactly {sorted(required_keys)}"
        if record.get("kind") != "preference":
            return f"record {index}: writer schema accepts preference records only"
        if record.get("attribute") != "delivery_city":
            return f"record {index}: attribute must be delivery_city"
        for key in ("entity", "attribute", "value", "source_event"):
            if not isinstance(record.get(key), str) or not record[key]:
                return f"record {index}: {key} must be a nonempty string"
    return None


def writer_prompt(case, history):
    """Full-memory generation. No correct continuation is supplied here."""
    return (
        "You are the memory writer for an order-processing assistant.\n"
        "Read the events below and return ONLY a JSON list of preference records.\n"
        'Each record has exactly these fields: {"kind":"preference",'
        '"entity":"customer name","attribute":"delivery_city",'
        '"value":"city","source_event":"supporting event id"}.\n'
        "Use the latest explicit preference event for each entity and attribute. "
        "Keep facts for different customers separate. Preserve source_event. "
        "Ignore metadata. Actions and tool observations report what happened; "
        "they do not establish customer preferences. Never infer a preference "
        "from an order destination.\n"
        "A previous_memory event contains a previously stored record. Retain it "
        "unless an explicit later preference event updates the same entity and "
        "attribute. Do not treat it as newly verified evidence.\n"
        "Put the delivery_city record for " + canonical_json(case["target_name"]) +
        " first if present, then the remaining preference records. "
        "Do not add explanatory text or Markdown fences.\n"
        "EVENTS_JSON:\n" + canonical_json(history) + "\nMEMORY_JSON:\n"
    )


def score_memory(case, history, memory, valid):
    """Score against explicit input evidence, never against observed actions.

    `omission` means at least one required entity/attribute is absent. Wrong
    values count as unsupported, not omissions. Duplicates are reported
    separately; target_correct requires exactly one correct target record.
    With previous-memory-only history there is no verified reference, so
    fact metrics are None; run_online attaches separate oracle-only scores.
    """
    reference = _latest_preference_records(history)
    records = memory if valid and isinstance(memory, list) else []
    by_key = {}
    for record in records:
        key = (record["entity"], record["attribute"])
        by_key.setdefault(key, []).append(record)
    target_key = (case["target_name"], "delivery_city")
    target_reference = reference.get(target_key)
    target_records = by_key.get(target_key, [])
    condition = case.get("_condition", "distractor")
    distractor = distractor_city(case, condition)
    has_reference = bool(reference)
    missing = sum(key not in by_key for key in reference)
    unsupported = sum(
        key not in reference or record["value"] != reference[key]["value"]
        for key, entries in by_key.items() for record in entries
    ) if has_reference else None
    correct = sum(
        len(by_key.get(key, [])) == 1 and
        by_key[key][0]["value"] == gold["value"]
        for key, gold in reference.items()
    ) if has_reference else None
    source_mismatch = sum(
        key in reference and record["source_event"] != reference[key]["source_event"]
        for key, entries in by_key.items() for record in entries
    ) if has_reference else None
    return {
        "target_distractor_error": (
            any(r["value"] == distractor for r in target_records) and
            distractor != target_reference["value"]
        ) if target_reference is not None else None,
        "target_correct": (
            bool(valid) and len(target_records) == 1 and
            target_records[0]["value"] == target_reference["value"]
        ) if target_reference is not None else None,
        "omission": bool(missing) if has_reference else None,
        "target_omission": not target_records if target_reference is not None else None,
        "missing_count": missing if has_reference else None,
        "unsupported_count": unsupported,
        "source_mismatch_count": source_mismatch,
        "duplicate_count": sum(max(0, len(entries) - 1) for entries in by_key.values()),
        "format_failure": not bool(valid),
        "required_count": len(reference),
        "record_count": len(records),
        "correct_count": correct,
        "reference_available": target_reference is not None,
    }


def run_writer(backend, case, history, heads=(), cfg=None):
    cfg = cfg or {}
    evidence = copy.deepcopy(history)
    evidence_hash = stable_hash(evidence)
    prompt = writer_prompt(case, evidence)
    generation = backend.generate(
        prompt, role="writer", heads=tuple(tuple(h) for h in heads),
        max_new_tokens=cfg.get("writer_max_new_tokens", cfg.get("max_new_tokens_writer", 384)),
        use_cache=cfg.get("use_cache", True),
    )
    raw = generation["text"]
    memory, parse_error = parse_json_output(raw, expected_type=list)
    schema_error = None if parse_error else _memory_schema_error(memory)
    error = parse_error or schema_error
    valid = error is None
    # Invalid generation is retained verbatim; no truth-based repair is applied.
    memory = memory if valid else None
    if stable_hash(history) != evidence_hash:
        raise RuntimeError("Writer mutated its supplied history")
    return {
        "memory": memory, "valid": valid, "raw": raw, "error": error,
        "metrics": score_memory(case, evidence, memory, valid),
        "token_count": generation.get("token_count", 0),
        "prompt_tokens": generation.get("prompt_tokens", 0),
        "seconds": generation.get("seconds", 0.0),
        "stop_reason": generation.get("stop_reason"),
        "history_hash": evidence_hash, "prompt_hash": stable_hash(prompt),
    }


def writer_snapshot(case, condition, history=None):
    """Teacher-forced target-field prefix, separate from free generation."""
    history = frozen_history(case, condition) if history is None else copy.deepcopy(history)
    reference = _latest_preference_records(history)
    target_key = (case["target_name"], "delivery_city")
    if target_key not in reference:
        raise ValueError("A writer search snapshot requires explicit target preference evidence")
    # Prefix ends immediately before the value, not at a generic last token.
    value_prefix = ('[{"kind":"preference","entity":' +
                    canonical_json(case["target_name"]) +
                    ',"attribute":"delivery_city","value":"')
    return {
        "source_id": case["source_id"], "split": case["split"],
        "role": "writer", "prompt": writer_prompt(case, history) + value_prefix,
        "gold": reference[target_key]["value"],
        "distractor": distractor_city(case, condition), "condition": condition,
        "history_hash": stable_hash(history),
        "template_id": case.get("template_id"),
    }


# %% Fixed-history Writer intervention and paired memory construction

def build_memory_pair(backend, case, history, write_heads, cfg):
    fixed_history = copy.deepcopy(history)
    tau_hash = stable_hash(fixed_history)
    results = {}
    for label, heads in (("M0", ()), ("M1", write_heads)):
        payload = {
            "source_id": case["source_id"], "condition": case.get("_condition", "distractor"),
            "history": fixed_history, "heads": [list(h) for h in heads],
            "prompt": writer_prompt(case, fixed_history), "cfg": cfg,
        }
        key = case["source_id"] + "-" + label + "-" + stable_hash(payload)[:16]
        result = checkpoint_item(
            "writer_pairs", key, payload,
            lambda heads=heads: run_writer(backend, case, copy.deepcopy(fixed_history), heads, cfg),
        )
        if result.get("history_hash") != tau_hash:
            raise RuntimeError("Cached Writer result has a different history")
        results[label] = result
    if stable_hash(history) != tau_hash or stable_hash(fixed_history) != tau_hash:
        raise RuntimeError("Fixed-history pairing changed the source trajectory")
    if results["M0"]["prompt_hash"] != results["M1"]["prompt_hash"]:
        raise RuntimeError("Writer arms received different prompts")
    return {
        "source_id": case["source_id"], "split": case["split"],
        "condition": case.get("_condition", "distractor"), "tau_hash": tau_hash,
        "write_heads": [list(h) for h in write_heads], **results,
    }


def _followup_case_and_environment(case, index):
    """Different order requests, matched exactly across all four arms."""
    followup = copy.deepcopy(case)
    initial = copy.deepcopy(case["initial_env"])
    initial["reservations"] = {}
    initial["orders"] = []
    initial["next_reservation"] = 1
    initial["next_order"] = 1
    # Every third task requests the other item first. City memory is unchanged.
    if index % 3 == 2:
        followup["preferred_item"], followup["backup_item"] = (
            followup["backup_item"], followup["preferred_item"])
    initial["stock"][followup["preferred_item"]] = 0 if index % 3 == 1 else 1 + index // 3
    initial["stock"][followup["backup_item"]] = 1 + index
    followup["initial_env"] = copy.deepcopy(initial)
    followup["followup_id"] = f'{case["source_id"]}:followup:{index}'
    return followup, initial


def evaluate_factorial(backend, case, pair, read_heads, cfg):
    """Four arms x independent followups; memory is read-only in this test."""
    if pair["source_id"] != case["source_id"]:
        raise ValueError("Memory pair and Actor case have different source IDs")
    if pair["M0"]["history_hash"] != pair["tau_hash"] or pair["M1"]["history_hash"] != pair["tau_hash"]:
        raise ValueError("Writer pair does not share the fixed trajectory")
    count = int(cfg.get("followup_tasks", 3))
    if count < 1:
        raise ValueError("followup_tasks must be positive")
    rows = []
    for followup_index in range(count):
        followup, initial = _followup_case_and_environment(case, followup_index)
        followup["_condition"] = pair["condition"]
        actor_cfg = {**cfg, "condition": pair["condition"]}
        for writer_intervention in (0, 1):
            result_memory = pair[f"M{writer_intervention}"]
            for actor_intervention in (0, 1):
                heads = read_heads if actor_intervention else ()
                arm = f"Y{writer_intervention}{actor_intervention}"
                memory = copy.deepcopy(result_memory["memory"])
                before = stable_hash(memory)
                payload = {
                    "source_id": case["source_id"], "followup": followup,
                    "initial_env": initial, "memory": memory,
                    "memory_valid": result_memory["valid"], "heads": [list(h) for h in heads],
                    "tau_hash": pair["tau_hash"], "cfg": cfg,
                }
                key = f'{case["source_id"]}-{arm}-{followup_index}-' + stable_hash(payload)[:16]
                if result_memory["valid"]:
                    actor_result = checkpoint_item(
                        "factorial", key, payload,
                        lambda: run_actor(backend, copy.deepcopy(followup), copy.deepcopy(memory),
                                          heads=heads, cfg=actor_cfg, initial_env=copy.deepcopy(initial)),
                    )
                    success = bool(actor_result.get("task_success", actor_result.get("success", False)))
                    failure_reason = None
                else:
                    # Parsing failures are full-pipeline failures, not dropped samples.
                    actor_result, success, failure_reason = None, False, "invalid_writer_memory"
                if stable_hash(memory) != before:
                    raise RuntimeError("Factorial evaluation modified its read-only memory")
                rows.append({
                    "source_id": case["source_id"], "split": case["split"],
                    "followup_id": followup["followup_id"], "followup_index": followup_index,
                    "arm": arm, "writer_intervention": writer_intervention,
                    "actor_intervention": actor_intervention,
                    "success": success, "task_success": success,
                    "distractor_action": bool(actor_result and actor_result.get("distractor_action", False)),
                    "format_failure": bool(not result_memory["valid"] or
                                           (actor_result and actor_result.get("format_failure", False))),
                    "memory_valid": result_memory["valid"], "memory_hash": before,
                    "environment_hash": stable_hash(initial), "tau_hash": pair["tau_hash"],
                    "failure_reason": failure_reason, "actor_result": actor_result,
                })
    return rows


# %% Local factual-value swaps: conditional diagnostic, all other fields fixed

def local_entry_variants(case, pair):
    """Swap only target.value. Preserve order, provenance, and all other bytes.

    Eligibility is conditioned on the observed M0/M1 values; report its rate.
    This diagnostic must not replace the unconditional whole-memory test.
    """
    natural = {label: copy.deepcopy(pair[label]["memory"]) for label in ("M0", "M1")}
    output = {"source_id": case["source_id"], "eligible": False,
              "reason": None, "variants": natural,
              "intervention_fields": ["value"], "provenance_held_fixed": True}
    if not all(pair[label]["valid"] for label in ("M0", "M1")):
        output["reason"] = "invalid_memory"
        return output
    indices = {}
    for label, memory in natural.items():
        matches = [i for i, r in enumerate(memory)
                   if r.get("kind") == "preference" and r.get("entity") == case["target_name"]
                   and r.get("attribute") == "delivery_city"]
        if len(matches) != 1:
            output["reason"] = f"{label}_target_missing_or_duplicate"
            return output
        indices[label] = matches[0]
    condition = pair.get("condition", case.get("_condition", "distractor"))
    base_value = natural["M0"][indices["M0"]]["value"]
    donor_value = natural["M1"][indices["M1"]]["value"]
    if base_value != distractor_city(case, condition) or base_value == expected_city(case, condition):
        output["reason"] = "M0_is_not_a_distractor_error"
        return output
    if donor_value != expected_city(case, condition):
        output["reason"] = "M1_is_not_a_correct_donor"
        return output
    corrected, corrupted = copy.deepcopy(natural["M0"]), copy.deepcopy(natural["M1"])
    corrected[indices["M0"]]["value"] = donor_value
    corrupted[indices["M1"]]["value"] = base_value
    output["variants"].update(M0_corrected=corrected, M1_corrupted=corrupted)
    output.update(eligible=True, reason=None, target_indices=indices,
                  token_check_required=True)
    return output


def evaluate_local_variants(backend, case, pair, read_heads=(), cfg=None):
    cfg = cfg or {}
    variants = local_entry_variants(case, pair)
    if not variants["eligible"]:
        return {**variants, "rows": []}
    rows = []
    for index in range(int(cfg.get("followup_tasks", 3))):
        followup, initial = _followup_case_and_environment(case, index)
        followup["_condition"] = pair["condition"]
        actor_cfg = {**cfg, "condition": pair["condition"]}
        for name, memory in variants["variants"].items():
            payload = {"case": followup, "initial_env": initial, "memory": memory,
                       "heads": [list(h) for h in read_heads], "cfg": cfg}
            key = f'{case["source_id"]}-{name}-{index}-' + stable_hash(payload)[:16]
            result = checkpoint_item(
                "local_swaps", key, payload,
                lambda: run_actor(backend, copy.deepcopy(followup), copy.deepcopy(memory),
                                  heads=read_heads, cfg=actor_cfg, initial_env=copy.deepcopy(initial)),
            )
            rows.append({"source_id": case["source_id"], "variant": name,
                         "followup_id": followup["followup_id"], "followup_index": index,
                         "task_success": bool(result.get("task_success", result.get("success", False))),
                         "actor_result": result, "environment_hash": stable_hash(initial)})
    return {**variants, "rows": rows}


# %% Optional online updating: separate from the read-only factorial endpoint

def run_online(backend, case, initial_memory, read_heads=(), write_heads=(), horizon=3, cfg=None):
    """Sequential Actor -> Writer updates with no injected ground truth.

    Orders are independent episodes; memory carries across them. Only the
    previous stored records plus actually observed actions/feedback enter the
    next Writer prompt. Oracle preference events are used for scoring only.
    """
    cfg = cfg or {}
    if horizon not in (1, 3, 5):
        raise ValueError("The prespecified exploratory horizons are 1, 3, and 5")
    error = _memory_schema_error(initial_memory)
    if error:
        return {"source_id": case["source_id"], "horizon": horizon, "steps": [],
                "failed": True, "failure_reason": "invalid_initial_memory: " + error}
    condition = case.get("_condition", "distractor")
    actor_cfg = {**cfg, "condition": condition}
    evaluator_reference = frozen_history(case, condition)
    current = copy.deepcopy(initial_memory)
    steps = []
    for index in range(horizon):
        followup, initial = _followup_case_and_environment(case, index)
        input_memory = copy.deepcopy(current)
        actor_result = run_actor(backend, followup, copy.deepcopy(current), heads=read_heads,
                                 cfg=actor_cfg, initial_env=copy.deepcopy(initial))
        # Full frozen_history would reintroduce truth. Include only real trace
        # events after filtering out the original preference/metadata records.
        actual_trace = [event for event in frozen_history(followup, condition, actor_result)
                        if event.get("kind") in ("observed_action", "tool_observation")]
        history = [{"kind": "previous_memory", "record": copy.deepcopy(record),
                    "sequence": j} for j, record in enumerate(current)]
        for event in actual_trace:
            event = copy.deepcopy(event)
            event["sequence"] = len(history)
            history.append(event)
        writer_result = run_writer(backend, followup, history, heads=write_heads, cfg=cfg)
        external_scores = score_memory(followup, evaluator_reference,
                                       writer_result["memory"], writer_result["valid"])
        steps.append({"source_id": case["source_id"], "step": index + 1,
                      "input_memory": input_memory, "actor_result": actor_result,
                      "writer_history": history, "writer_result": writer_result,
                      "external_reference_metrics": external_scores})
        if not writer_result["valid"]:
            return {"source_id": case["source_id"], "horizon": horizon, "steps": steps,
                    "failed": True, "failure_reason": "invalid_writer_memory", "completed_steps": index + 1}
        current = copy.deepcopy(writer_result["memory"])
    return {"source_id": case["source_id"], "horizon": horizon, "steps": steps,
            "failed": False, "failure_reason": None, "completed_steps": horizon,
            "final_memory": current}

In [8]:
# %% Writer evaluation stages: actual fixed experiences, unconditional source sets

def collect_fixed_histories(backend, cases, conditions, cfg):
    """Run an untreated Actor once per case/condition, then freeze its experience.

    Failed, malformed, and successful episodes are all retained. Every Writer
    arm later receives exactly this recorded history; no Actor is rerun under
    Writer interventions. Checkpoints retain the full Actor result separately.
    """
    histories = {condition: {} for condition in conditions}
    for condition in conditions:
        for original_case in cases:
            case = copy.deepcopy(original_case)
            case["_condition"] = condition
            actor_cfg = {**cfg, "condition": condition}
            memory = memory_for_condition(case, condition)
            payload = {"case": case, "memory": memory, "heads": [],
                       "cfg": actor_cfg, "purpose": "fixed_writer_history"}
            key = f'{case["source_id"]}-{condition}-' + stable_hash(payload)[:16]
            actor_result = checkpoint_item(
                "fixed_history_actors", key, payload,
                lambda: run_actor(backend, copy.deepcopy(case), copy.deepcopy(memory),
                                  heads=(), cfg=actor_cfg,
                                  initial_env=copy.deepcopy(case["initial_env"])),
            )
            histories[condition][case["source_id"]] = frozen_history(case, condition, actor_result)
    return histories


def evaluate_writer(backend, cases, histories, head_sets, cfg,
                    conditions=("correct_only", "distractor", "matched_neutral",
                                "distractor_value_swap", "useful_memory", "legitimate_update")):
    """All planned source cases x conditions x named interventions.

    head_sets is a dict {label: [(layer, head), ...]}; include an untreated
    baseline explicitly. Format failures stay in the output denominator.
    Metrics are flattened for source-level paired summaries, while raw text,
    parse errors, tokens, and full memory remain in writer_result.
    """
    if not isinstance(head_sets, dict) or not head_sets:
        raise ValueError("head_sets must be a nonempty dict of named interventions")
    rows = []
    for condition in conditions:
        if condition not in histories:
            raise KeyError(f"No frozen histories for condition {condition}")
        for original_case in cases:
            case = copy.deepcopy(original_case)
            case["_condition"] = condition
            source_id = case["source_id"]
            if source_id not in histories[condition]:
                raise KeyError(f"Missing frozen history: {condition}/{source_id}")
            history = copy.deepcopy(histories[condition][source_id])
            history_hash = stable_hash(history)
            writer_cfg = {**cfg, "condition": condition}
            for label, heads in head_sets.items():
                payload = {"case": case, "history": history,
                           "heads": [list(h) for h in heads], "cfg": writer_cfg,
                           "prompt": writer_prompt(case, history)}
                key = f"{source_id}-{condition}-{label}-" + stable_hash(payload)[:16]
                result = checkpoint_item(
                    "writer_evaluation", key, payload,
                    lambda: run_writer(backend, copy.deepcopy(case), copy.deepcopy(history),
                                       heads=heads, cfg=writer_cfg),
                )
                if result["history_hash"] != history_hash:
                    raise RuntimeError("Writer evaluation did not use the frozen history")
                rows.append({
                    "source_id": source_id, "split": case["split"],
                    "condition": condition, "intervention": str(label),
                    **result["metrics"], "valid": result["valid"],
                    "token_count": result.get("token_count", 0),
                    "seconds": result.get("seconds", 0.0),
                    "history_hash": history_hash, "writer_result": result,
                })
            if stable_hash(history) != history_hash:
                raise RuntimeError("Writer evaluation modified frozen history")
    return rows


def evaluate_memory_pipeline(backend, cases, histories, read_heads, write_heads, cfg):
    """Unconditional whole-memory endpoint plus a labeled local diagnostic.

    Returns pairs as a source-ID dictionary, factorial as flat rows over every
    case, and local as per-source dictionaries including ineligible cases.
    Local eligible-only results never filter the primary factorial sample.
    """
    condition = "distractor"
    if condition not in histories:
        raise KeyError("Memory pipeline requires distractor frozen histories")
    pairs, factorial, local = {}, [], []
    for original_case in cases:
        case = copy.deepcopy(original_case)
        case["_condition"] = condition
        source_id = case["source_id"]
        if source_id not in histories[condition]:
            raise KeyError(f"Missing distractor history for {source_id}")
        history = copy.deepcopy(histories[condition][source_id])
        pair_cfg = {**cfg, "condition": condition}
        pair = build_memory_pair(backend, case, history, write_heads, pair_cfg)
        pairs[source_id] = pair
        factorial.extend(evaluate_factorial(backend, case, pair, read_heads, pair_cfg))
        # Untreated downstream Actor isolates the local memory edit itself.
        local.append(evaluate_local_variants(backend, case, pair, read_heads=(), cfg=pair_cfg))
    return {"pairs": pairs, "factorial": factorial, "local": local}

###  Assessment and statistics 

 Main analysis of retention of all planning cases;  token  Unqualified only skips  margin。 Random matching each layer  head  Count and select  heads  Do not overlap. If you cannot construct this random comparison, the  K  Do not enter the development pool. 

bootstrap  The independent unit is...  source episode， Well start with the same thing.  source  Inner aggregation of follow-up tasks and matching; cannot be used  3   followups  Search or more  seed  As a stand-alone sample. Intersection is descriptive.  percentile bootstrap； Formal inference is that sufficient samples are required to match presets. 

In [9]:
# Read-side evaluation and source-clustered descriptive/paired uncertainty.
import collections
import math
import random
import numpy as np


def _scalar_float(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    if hasattr(value, "item"):
        value = value.item()
    result = float(value)
    if not math.isfinite(result):
        raise ValueError("Nonfinite measurement; preserve the failing case and diagnose it")
    return result


def evaluate_read(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    """Execute every planned source × condition × intervention, with checkpoints.

    head_sets maps human-readable labels to (layer, head) pairs. Actual full
    behavior is never filtered by token eligibility or initial attack success.
    Fixed-prefix margin is a secondary, explicitly eligible-only diagnostic.
    cfg is expected to contain model/revision/dtype/decoding identity so the
    checkpoint payload changes when the run configuration changes.
    """
    if not hasattr(head_sets, "items") or not head_sets:
        raise ValueError("head_sets must be a nonempty mapping label -> head pairs")
    rows = []
    for case in cases:
        for condition in conditions:
            memory = memory_for_condition(case, condition)
            snapshot = actor_snapshot(case, condition)
            gold_id = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
            distractor_id = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
            eligible = gold_id is not None and distractor_id is not None and gold_id != distractor_id
            for intervention, selected in head_sets.items():
                heads = [list(pair) for pair in selected]
                condition_cfg = dict(cfg, condition=condition)
                payload = {"case": case, "memory": memory, "heads": heads,
                           "cfg": condition_cfg, "intervention": intervention,
                           "snapshot": snapshot, "version": "read_eval_v1"}

                def compute(case=case, memory=memory, heads=heads,
                            condition_cfg=condition_cfg, snapshot=snapshot,
                            intervention=intervention, eligible=eligible,
                            gold_id=gold_id, distractor_id=distractor_id):
                    result = run_actor(backend, case, memory, heads=heads, cfg=condition_cfg)
                    result.update({"intervention": intervention, "heads": heads,
                                   "single_token_eligible": eligible,
                                   "gold_token_id": gold_id, "distractor_token_id": distractor_id,
                                   "snapshot_margin": _scalar_float(backend.margin(snapshot, heads=heads)) if eligible else None,
                                   "snapshot_origin": snapshot["snapshot_origin"]})
                    return result

                key = f"{case['source_id']}-{condition}-{intervention}"
                rows.append(checkpoint_item("read_eval", key, payload, compute))
    return rows


def same_layer_random_heads(selected, n_layers, n_heads, seed):
    """Match each selected layer's head count, excluding ALL selected heads.

    Raise if the requested disjoint control cannot be formed. Never silently
    relax exclusion or layer matching. Repeat with distinct recorded seeds.
    """
    selected = [tuple(pair) for pair in selected]
    if len(set(selected)) != len(selected):
        raise ValueError("selected contains duplicate heads")
    if n_layers < 1 or n_heads < 1:
        raise ValueError("n_layers and n_heads must be positive")
    for layer, head in selected:
        if type(layer) is not int or type(head) is not int or not (0 <= layer < n_layers and 0 <= head < n_heads):
            raise ValueError(f"Invalid selected head {(layer, head)}")
    by_layer = collections.defaultdict(set)
    for layer, head in selected:
        by_layer[layer].add(head)
    rng = random.Random(int(seed))
    control = []
    for layer in sorted(by_layer):
        available = [head for head in range(n_heads) if head not in by_layer[layer]]
        count = len(by_layer[layer])
        if len(available) < count:
            raise ValueError(f"Layer {layer}: {count} selected heads but only {len(available)} nonselected heads; no disjoint layer-matched control")
        control.extend((layer, head) for head in rng.sample(available, count))
    return sorted(control)


def actor_diagnostics(rows):
    """Descriptive process failures; never remove episodes or alter scoring."""
    diagnostics = []
    for row in rows:
        errors = [event["observation"].get("error") for event in row["events"]
                  if isinstance(event.get("observation"), dict)
                  and not event["observation"].get("ok", False)]
        reserved = sum(event.get("action", {}).get("tool") == "reserve_item"
                       and bool((event.get("observation") or {}).get("ok"))
                       for event in row["events"] if isinstance(event.get("action"), dict))
        diagnostics.append({
            key: row[key] for key in ("source_id", "condition", "intervention", "template_id",
                                     "preferred_available", "position_order", "task_success",
                                     "format_failure", "terminal", "action_count", "budget_exhausted")
        })
        diagnostics[-1].update(tool_error_count=len(errors),
                               out_of_stock_count=errors.count("out_of_stock"),
                               invalid_reservation_count=errors.count("invalid_reservation"),
                               successful_reservations=reserved,
                               accepted_order_count=row["accepted_order_count"],
                               gold_city=row["gold_city"], gold_item=row["gold_item"])
    return diagnostics


def _passes_filters(row, filters):
    if filters is None:
        return True
    if callable(filters):
        return bool(filters(row))
    for key, expected in filters.items():
        actual = row.get(key)
        if isinstance(expected, (list, tuple, set, frozenset)):
            if actual not in expected:
                return False
        elif actual != expected:
            return False
    return True


def _get_metric(row, metric):
    if metric in row:
        return row[metric]
    return row.get("metrics", {}).get(metric)


def _source_bootstrap(values, seed, n_boot):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("Bootstrap requires a nonempty finite 1-D source-level array")
    if not isinstance(n_boot, int) or n_boot < 1:
        raise ValueError("n_boot must be a positive integer")
    rng = np.random.default_rng(int(seed))
    # Chunk so large source counts do not allocate n_boot × n_sources at once.
    estimates = []
    for start in range(0, n_boot, 256):
        count = min(256, n_boot - start)
        indices = rng.integers(0, len(values), size=(count, len(values)))
        estimates.extend(values[indices].mean(axis=1).tolist())
    lower, upper = np.quantile(np.asarray(estimates), [0.025, 0.975])
    return {"estimate": float(values.mean()), "ci_low": float(lower), "ci_high": float(upper),
            "n_sources": int(len(values)), "n_boot": n_boot,
            "ci_method": "percentile_source_bootstrap_95pct",
            "small_sample_warning": bool(len(values) < 20)}


def paired_effect(rows, arm_col, arm0, arm1, metric="task_success", filters=None, seed=11, n_boot=2000):
    """Mean(arm1 − arm0), bootstrapping source cases after within-source means.

    Repeated future tasks, fields, or seeds never become independent units here.
    A missing arm/source/metric is an error, not silent complete-case filtering.
    Equal repeat counts are required within each paired source; if followup_id
    exists, its set must also match. Use filters to select the planned contrast.
    """
    if arm0 == arm1:
        raise ValueError("arm0 and arm1 must differ")
    grouped = {arm0: collections.defaultdict(list), arm1: collections.defaultdict(list)}
    for row in rows:
        arm = row.get(arm_col)
        if arm not in grouped or not _passes_filters(row, filters):
            continue
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        value = _get_metric(row, metric)
        if value is None:
            raise ValueError(f"Missing {metric} for {row['source_id']} / {arm}")
        grouped[arm][row["source_id"]].append((row, _scalar_float(value)))
    sources0, sources1 = set(grouped[arm0]), set(grouped[arm1])
    if not sources0 or sources0 != sources1:
        raise ValueError(f"Unpaired or empty sources: arm0={len(sources0)}, arm1={len(sources1)}, missing_in_arm1={sorted(sources0-sources1)[:5]}, missing_in_arm0={sorted(sources1-sources0)[:5]}")
    differences, means0, means1 = [], [], []
    count_rows = 0
    for source in sorted(sources0):
        entries0, entries1 = grouped[arm0][source], grouped[arm1][source]
        if len(entries0) != len(entries1):
            raise ValueError(f"Unmatched repeat counts for source {source}: {len(entries0)} vs {len(entries1)}")
        if any("followup_id" in row for row, _ in entries0 + entries1):
            ids0 = collections.Counter(row.get("followup_id") for row, _ in entries0)
            ids1 = collections.Counter(row.get("followup_id") for row, _ in entries1)
            if ids0 != ids1:
                raise ValueError(f"Unpaired followup_id values for source {source}")
        mean0 = sum(v for _, v in entries0) / len(entries0)
        mean1 = sum(v for _, v in entries1) / len(entries1)
        means0.append(mean0)
        means1.append(mean1)
        differences.append(mean1 - mean0)
        count_rows += len(entries0) + len(entries1)
    result = _source_bootstrap(differences, seed, n_boot)
    result.update({"arm_col": arm_col, "arm0": arm0, "arm1": arm1, "metric": metric,
                   "direction": "arm1_minus_arm0", "n_rows": count_rows,
                   "arm0_mean": float(np.mean(means0)), "arm1_mean": float(np.mean(means1)),
                   "n_unpaired_sources": 0, "seed": int(seed)})
    return result


def bootstrap_summary(rows, group_cols, metrics, n_boot=2000):
    """Descriptive group means with a source-clustered 95% percentile interval.

    Returns one row per group × metric. Missing optional metrics (e.g. ineligible
    snapshot margins) are counted explicitly. For confirmatory arm comparisons,
    use paired_effect; these separate intervals do not estimate a difference.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if isinstance(metrics, str):
        metrics = [metrics]
    groups = collections.defaultdict(list)
    for row in rows:
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        key = tuple(row.get(col) for col in group_cols)
        groups[key].append(row)
    summaries = []
    for group_index, (key, members) in enumerate(sorted(groups.items(), key=lambda item: repr(item[0]))):
        for metric_index, metric in enumerate(metrics):
            per_source = collections.defaultdict(list)
            n_missing = 0
            for row in members:
                value = _get_metric(row, metric)
                if value is None:
                    n_missing += 1
                else:
                    per_source[row["source_id"]].append(_scalar_float(value))
            summary = dict(zip(group_cols, key))
            summary.update({"metric": metric, "n_rows": len(members),
                            "n_missing_rows": n_missing, "n_valid_rows": len(members) - n_missing,
                            "n_all_sources": len({row["source_id"] for row in members})})
            values = [sum(values) / len(values) for _, values in sorted(per_source.items())]
            if values:
                summary.update(_source_bootstrap(values, 11 + 1009 * group_index + metric_index, n_boot))
            else:
                summary.update({"estimate": None, "ci_low": None, "ci_high": None,
                                "n_sources": 0, "n_boot": n_boot, "ci_method": "no_valid_values"})
            summaries.append(summary)
    return summaries

In [10]:
# Run gates, final protocol, and serializable result exports.
import inspect
import csv


def _implementation_source(obj):
    # Interactive notebook classes lack a normal module file. Their method
    # functions do retain IPython linecache source and are stable on reruns.
    if inspect.isclass(obj):
        return {"name": obj.__name__, "doc": obj.__doc__,
                "bases": [base.__qualname__ for base in obj.__bases__],
                "methods": {name: inspect.getsource(method) for name, method in vars(obj).items()
                            if inspect.isfunction(method)}}
    return inspect.getsource(obj)


def live_implementation_hash():
    return stable_hash({"definitions": {name: _implementation_source(globals()[name]) for name in IMPLEMENTATION_NAMES},
                        "task_templates": TASK_TEMPLATES, "conditions": CONDITIONS})


def verify_live_run():
    current_hash = live_implementation_hash()
    current_run = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": current_hash})
    if current_hash != IMPLEMENTATION_HASH or current_run != RUN_FINGERPRINT:
        raise RuntimeError("Live definitions/constants/config differ from the run manifest. Start a new run rather than mixing checkpoints.")


def capability_gate(rows, cfg):
    clean = [r for r in rows if r["condition"] == "correct_only" and r["intervention"] == "none"]
    if not clean:
        raise ValueError("Capability pilot requires untreated correct_only cases")
    success = float(np.mean([r["task_success"] for r in clean]))
    format_failure = float(np.mean([r["format_failure"] for r in clean]))
    return {"eligible": success >= cfg["min_pilot_success"] and format_failure <= cfg["max_pilot_format_failure"],
            "n_sources": len(clean), "clean_success": success, "format_failure": format_failure,
            "min_success": cfg["min_pilot_success"], "max_format_failure": cfg["max_pilot_format_failure"]}


def natural_writer_gate(rows, cfg):
    clean = [r for r in rows if r["condition"] == "correct_only" and r["intervention"] == "none"]
    attacks = [r for r in rows if r["condition"] == "distractor" and r["intervention"] == "none"]
    if not clean or not attacks:
        raise ValueError("Writer pilot requires paired clean/distractor conditions")
    errors = sum(bool(r["target_distractor_error"]) for r in attacks)
    valid = float(np.mean([not r["format_failure"] for r in clean]))
    correct = float(np.mean([r["target_correct"] for r in clean]))
    return {"eligible": errors >= cfg["min_natural_writer_errors"] and valid >= cfg["min_writer_valid"] and correct >= cfg["min_pilot_success"],
            "natural_distractor_errors": errors, "n_sources": len(attacks),
            "clean_valid_fraction": valid, "clean_target_correct": correct,
            "min_errors": cfg["min_natural_writer_errors"],
            "interpretation": "feasibility_gate_only_not_a_significance_test"}


def _atomic_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = sorted({key for row in rows for key in row})
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: canonical_json(value) if isinstance(value, (list, dict, tuple)) else value
                             for key, value in row.items()})
    os.replace(temporary, path)


def export_rows(name, rows, group_cols, metrics):
    atomic_json(RUN_DIR / "results" / f"{name}-raw.json", rows)
    summary = bootstrap_summary(rows, group_cols, metrics, n_boot=CFG["bootstrap_reps"])
    _atomic_csv(RUN_DIR / "results" / f"{name}-summary.csv", summary)
    atomic_json(RUN_DIR / "results" / f"{name}-summary.json", summary)
    return summary


def factorial_contrasts(rows):
    if not rows:
        return []
    definitions = [("writer_effect_actor_untreated", "Y00", "Y10"),
                   ("reader_effect_baseline_memory", "Y00", "Y01"),
                   ("combined_effect", "Y00", "Y11"),
                   ("reader_effect_intervened_memory", "Y10", "Y11")]
    effects = [dict(contrast=name, **paired_effect(rows, "arm", a, b, n_boot=CFG["bootstrap_reps"]))
               for name, a, b in definitions]
    # Interaction = (Y11-Y10) - (Y01-Y00), computed within the same source.
    by_source = {}
    for row in rows:
        by_source.setdefault(row["source_id"], {}).setdefault(row["arm"], []).append(float(row["task_success"]))
    interaction = []
    for source, arms in sorted(by_source.items()):
        if set(arms) != {"Y00", "Y01", "Y10", "Y11"}:
            raise ValueError(f"Incomplete factorial for {source}")
        means = {key: np.mean(values) for key, values in arms.items()}
        interaction.append(means["Y11"] - means["Y10"] - means["Y01"] + means["Y00"])
    effects.append(dict(contrast="writer_reader_interaction", **_source_bootstrap(interaction, 11, CFG["bootstrap_reps"])))
    return effects


def freeze_protocol(read_selection, write_selection, read_controls, write_controls, data_hashes, writer_gate):
    verify_live_run()
    if QUICK_MODE:
        raise RuntimeError("Final test is disabled in QUICK_MODE. Use a new full-run directory and finish development first.")
    protocol = {"schema": "agent_entrainment_v1", "run_fingerprint": RUN_FINGERPRINT,
                "implementation_hash": IMPLEMENTATION_HASH, "model": MODEL_IDENTITY,
                "config": CFG, "data_hashes": data_hashes,
                "primary_search_seed": CFG["search_seeds"][0],
                "read_heads": read_selection["heads"] if read_selection else [],
                "write_heads": write_selection["heads"] if write_selection else [],
                "read_controls": read_controls, "write_controls": write_controls,
                "writer_branch": bool(writer_gate and writer_gate["eligible"] and write_selection and write_selection["eligible"]),
                "writer_gate": writer_gate,
                "primary_endpoints": {
                    "read": {"condition": "distractor", "metric": "task_success", "contrast": "H_R minus none", "population": "all test sources"},
                    "write": {"condition": "distractor", "metric": "target_correct", "contrast": "H_W minus none", "population": "all test sources; only if writer branch prespecified"},
                    "propagation": {"condition": "distractor", "metric": "task_success", "contrast": "Y10 minus Y00", "population": "all test sources; only if both head sets available"}},
                "secondary_endpoints": ["distractor_action", "format_failure", "omission", "useful_memory", "legitimate_update", "random_head_controls", "Y11 minus Y00", "factorial_interaction"],
                "uncertainty_unit": "source_episode", "decoding": "greedy_complete_json",
                "test_rule": "no retuning or case exclusion after inspecting test outcomes"}
    # Deep JSON snapshot: no aliases to mutable CFG/heads and no tuple/list
    # mismatch when the same frozen protocol is loaded after a disconnect.
    protocol = json.loads(canonical_json(protocol))
    path = RUN_DIR / "frozen_protocol.json"
    if path.exists() and json.loads(path.read_text()) != protocol:
        raise RuntimeError("Frozen protocol differs from current settings/heads. Do not overwrite a used test protocol.")
    atomic_json(path, protocol)
    return protocol


def verify_protocol(protocol, data):
    verify_live_run()
    if protocol["run_fingerprint"] != RUN_FINGERPRINT or protocol["implementation_hash"] != IMPLEMENTATION_HASH:
        raise RuntimeError("Protocol/model/config/code mismatch")
    if protocol["data_hashes"] != {split: stable_hash(cases) for split, cases in data.items()}:
        raise RuntimeError("Data changed after protocol freeze")


def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [11]:
IMPLEMENTATION_NAMES = ['canonical_json', 'stable_hash', 'atomic_json', 'checkpoint_item', 'seed_everything', 'normalize_heads', 'ablation_hooks', 'TLBackend', 'eligible_snapshots', 'snapshot_pairs', 'gumbel_drop', '_save_search', 'train_mask', 'select_heads_dev', 'build_cases', '_preference_record', 'expected_city', 'distractor_city', 'memory_for_condition', '_json_no_duplicates', 'parse_json_output', 'OrderEnvironment', '_task_goal', '_actor_protocol_examples', 'actor_prompt', 'run_actor', 'frozen_history', 'actor_snapshot', '_latest_preference_records', '_memory_schema_error', 'writer_prompt', 'score_memory', 'run_writer', 'writer_snapshot', 'build_memory_pair', '_followup_case_and_environment', 'evaluate_factorial', 'local_entry_variants', 'evaluate_local_variants', 'run_online', 'collect_fixed_histories', 'evaluate_writer', 'evaluate_memory_pipeline', '_scalar_float', 'evaluate_read', 'same_layer_random_heads', 'actor_diagnostics', '_passes_filters', '_get_metric', '_source_bootstrap', 'paired_effect', 'bootstrap_summary', '_implementation_source', 'live_implementation_hash', 'verify_live_run', 'capability_gate', 'natural_writer_gate', '_atomic_csv', 'export_rows', 'factorial_contrasts', 'freeze_protocol', 'verify_protocol', 'runtime_smoke']

# IPython keeps executable cell sources. Hash actual live definitions, not just a version label.
IMPLEMENTATION_HASH = live_implementation_hash()
print("Implementation:", IMPLEMENTATION_HASH[:16])

Implementation: 157da2f9d371bc88


## 03 ·  Login, model loading and running list 

 Keep the old one.  `fold_ln=False`、`center_writing_weights=False`、`center_unembed=False`。 Do not automatically quantify and do not replace  Instruct。 To accommodate the full experience, display settings  `n_ctx=4096`（ The old version of the library will only be assigned to this model by default  2048）； Expand only the context buffer when running, with weights equal to  RoPE  parameters are retained. Reject overlong inputs without truncating evidence. The next step checks whether the base model can execute the complete tool protocol  pilot  decides. If core definitions or hash cells have not run, this cell reports the missing steps. 

In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [14]:
if not callable(globals().get("live_implementation_hash")) or "IMPLEMENTATION_NAMES" not in globals():
    raise RuntimeError(" Please run in sequence. 02 Section all defined blocks and their last  IMPLEMENTATION_NAMES/hash  block, rerun model loaded. ")
missing_definitions = [name for name in IMPLEMENTATION_NAMES if name not in globals()]
if missing_definitions:
    raise RuntimeError("02 Section is defined as not running: " + ", ".join(missing_definitions))
IMPLEMENTATION_HASH = live_implementation_hash()
if not get_token():
    notebook_login()
if not torch.cuda.is_available():
    raise RuntimeError(" This.  notebook   8B  Experimental needs.  CUDA GPU； First.  Colab  Change running time. ")
if REQUESTED_DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError(" current  GPU  Not supported  bfloat16。 Replaces running or is specified as  float16  and use new  RUN_NAME； Do not mix the results with precision. ")
DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[REQUESTED_DTYPE]
DEVICE = "cuda"
revision_path = RUN_DIR / "model_revision.json"
if revision_path.exists():
    saved_revision = json.loads(revision_path.read_text())
    if saved_revision["model_name"] != MODEL_NAME or (MODEL_REVISION and MODEL_REVISION != saved_revision["sha"]):
        raise RuntimeError("Model revision differs; use a new RUN_NAME.")
    RESOLVED_REVISION = saved_revision["sha"]
else:
    RESOLVED_REVISION = model_info(MODEL_NAME, revision=MODEL_REVISION).sha
    atomic_json(revision_path, {"model_name": MODEL_NAME, "sha": RESOLVED_REVISION})
MODEL_IDENTITY = {"name": MODEL_NAME, "revision": RESOLVED_REVISION, "dtype": REQUESTED_DTYPE,
                  "packages": PACKAGE_VERSIONS, "python": sys.version.split()[0]}
RUN_FINGERPRINT = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": IMPLEMENTATION_HASH})
manifest = {"fingerprint": RUN_FINGERPRINT, "config": CFG, "model": MODEL_IDENTITY,
            "implementation": IMPLEMENTATION_HASH}
manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    raise RuntimeError("Run directory contains another configuration or package stack. Choose a new RUN_NAME.")
atomic_json(manifest_path, manifest)
seed_everything(CFG["data_seed"])
print("GPU:", torch.cuda.get_device_name(0), "GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
model = HookedTransformer.from_pretrained(
    MODEL_NAME, revision=RESOLVED_REVISION, device=DEVICE, dtype=DTYPE, n_ctx=CFG["context_window"],
    fold_ln=False, center_writing_weights=False, center_unembed=False,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
N_LAYERS, N_HEADS = model.cfg.n_layers, model.cfg.n_heads
backend = TLBackend(model, CFG)
print("Model:", MODEL_NAME, "revision:", RESOLVED_REVISION, "layers/query-heads:", N_LAYERS, N_HEADS)

RuntimeError: Run directory contains another configuration or package stack. Choose a new RUN_NAME.

## 04 ·  Model Interface  smoke  and  token  Audit 

 Here, check it out.  GPU  Go on.  hook、mask  Gradient and  cache； Failed to fix interface first, not enter the thesis experiment. `hook_z`  and  `hook_result`  Low precision may introduce small numerical differences; record the maximum deviation against preset tolerances. Short cached and uncached  greedy  Generating must be consistent. 

In [13]:
SMOKE = runtime_smoke(backend)
atomic_json(RUN_DIR / "software_smoke.json", SMOKE)
display(SMOKE)

# Initial vocabulary screen is role-specific; exact task prefixes are checked again below.
TOKEN_AUDIT = []
for city in CFG["city_candidates"]:
    actor_id = backend.one_token_id('Next tool call:\n{"city":"', city)
    writer_id = backend.one_token_id('MEMORY_JSON:\n[{"value":"', city)
    TOKEN_AUDIT.append({"city": city, "actor_token": actor_id, "writer_token": writer_id,
                        "eligible": actor_id is not None and writer_id is not None})
CITY_VALUES = [row["city"] for row in TOKEN_AUDIT if row["eligible"]]
if len(CITY_VALUES) < 4:
    raise RuntimeError("Fewer than four single-token cities; inspect TOKEN_AUDIT before proceeding.")
atomic_json(RUN_DIR / "token_audit.json", TOKEN_AUDIT)
display(pd.DataFrame(TOKEN_AUDIT))

NameError: name 'backend' is not defined

## 05 ·  Create and Seal  source  Data 

 Pregenerated and saved  test  inputs, but do not use their model outputs during development. Diagnostic prefixes are manually constructed valid reserved states, not actual model trajectories; subsequent full behavioral tests are always executed by the model itself. 

In [14]:
DATA = {split: build_cases(split, CFG[f"n_{split}"], CFG["data_seed"], CITY_VALUES)
        for split in ("pilot", "train", "dev", "test")}
for left, right in (("train", "dev"), ("train", "test"), ("dev", "test"), ("pilot", "train"), ("pilot", "dev"), ("pilot", "test")):
    for key in ("source_id", "target_name", "other_name", "template_id"):
        assert not ({case[key] for case in DATA[left]} & {case[key] for case in DATA[right]})
DATA_HASHES = {split: stable_hash(cases) for split, cases in DATA.items()}
atomic_json(RUN_DIR / "data_manifest.json", {"hashes": DATA_HASHES, "data": DATA})
# Development token lengths quantify the approximately matched neutral control.
PROMPT_LENGTHS = [{"source_id": case["source_id"], "condition": condition,
                   "actor_prompt_tokens": len(backend.encode(actor_prompt(case, memory_for_condition(case, condition), [])))}
                  for case in DATA["dev"] for condition in CONDITIONS]
_atomic_csv(RUN_DIR / "results" / "dev-prompt-lengths.csv", PROMPT_LENGTHS)
DESIGN_ROWS = [{"split": split, "template_id": case["template_id"],
                "preferred_available": case["initial_env"]["stock"][case["preferred_item"]] > 0,
                "position_order": case["position_order"]}
               for split, cases in DATA.items() for case in cases]
_atomic_csv(RUN_DIR / "results" / "design-factors.csv", DESIGN_ROWS)
display(pd.DataFrame(DESIGN_ROWS).groupby(["split", "template_id", "preferred_available", "position_order"]).size().rename("n_sources").reset_index())
print("Source counts:", {split:len(cases) for split,cases in DATA.items()})
print("Test outputs have not been evaluated.")
display(pd.DataFrame(PROMPT_LENGTHS).groupby("condition")["actor_prompt_tokens"].agg(["mean", "min", "max"]))

,split,template_id,preferred_available,position_order,n_sources
0,dev,dev_request,False,other_first,1
1,dev,dev_request,False,target_first,1
2,dev,dev_request,True,other_first,1
3,dev,dev_request,True,target_first,1
4,dev,dev_shipment,False,other_first,1
5,dev,dev_shipment,False,target_first,1
6,dev,dev_shipment,True,other_first,1
7,dev,dev_shipment,True,target_first,1
8,pilot,pilot_dispatch,False,other_first,1
9,pilot,pilot_dispatch,False,target_first,1


Source counts: {'pilot': 8, 'train': 16, 'dev': 8, 'test': 24}
Test outputs have not been evaluated.


,mean,min,max
condition,,,
correct_only,757.5,754,761
distractor,798.5,795,802
distractor_value_swap,798.5,795,802
legitimate_update,798.5,795,802
matched_neutral,798.5,795,802
useful_memory,798.5,795,802


## 06 ·  First measure.  Actor  Capability, look at nature.  Writer  Error 

 Capacity  gate  Its only for feasibility judgements. Its not for visibility.  base  Model Company  clean JSON  if basic tool execution fails, save the run and skip subsequent search. Examples can be revised in a new development run / protocol; do not interpret format collapse as  entrainment。

v4  Autosave  `pilot-actor-diagnostics.csv`  and  `pilot-failures.json`， Show inventory error, invalid booking, number of successful bookings and completeness prompt。 The grouping and trajectories are first viewed; 8 source At least. 7 One success is enough to satisfy the original. 80% threshold. Passing the capability threshold is still not equal to observing entrainment。

Writer gate  Request  clean  Format / Target facts are basically available, and  distractor  At least observed in the experience.  2  natural misbindings. Observing none does not prove absence and does not justify manual contamination  M0  Fill it up. 

In [17]:
verify_live_run()
PILOT_READ = evaluate_read(backend, DATA["pilot"], {"none": []}, CFG,
                           conditions=("correct_only", "distractor"))
CAPABILITY = capability_gate(PILOT_READ, CFG)
atomic_json(RUN_DIR / "pilot_capability.json", CAPABILITY)
display(pd.DataFrame(export_rows("pilot-read", PILOT_READ, ["condition", "intervention"],
                                 ["task_success", "distractor_action", "format_failure", "stock_check_adherence"])))
display(CAPABILITY)
PILOT_DIAGNOSTICS = actor_diagnostics(PILOT_READ)
_atomic_csv(RUN_DIR / "results" / "pilot-actor-diagnostics.csv", PILOT_DIAGNOSTICS)
atomic_json(RUN_DIR / "results" / "pilot-failures.json", [row for row in PILOT_READ if not row["task_success"]])
display(pd.DataFrame(PILOT_DIAGNOSTICS))
print(" Failed original trajectory (with every step) prompt）：", RUN_DIR / "results" / "pilot-failures.json")
PILOT_WRITER, WRITER_GATE = [], None
if CAPABILITY["eligible"]:
    # Reuse the actually recorded untreated pilot traces, with no rerun or invented actions.
    PILOT_HISTORIES = {condition: {} for condition in ("correct_only", "distractor")}
    pilot_by_id = {case["source_id"]:case for case in DATA["pilot"]}
    for row in PILOT_READ:
        PILOT_HISTORIES[row["condition"]][row["source_id"]] = frozen_history(pilot_by_id[row["source_id"]], row["condition"], row)
    PILOT_WRITER = evaluate_writer(backend, DATA["pilot"], PILOT_HISTORIES, {"none": []}, CFG,
                                   conditions=("correct_only", "distractor"))
    WRITER_GATE = natural_writer_gate(PILOT_WRITER, CFG)
    display(pd.DataFrame(export_rows("pilot-writer", PILOT_WRITER, ["condition", "intervention"],
                                     ["target_distractor_error", "target_correct", "omission", "format_failure"])))
    display(WRITER_GATE)
else:
    print("Actor capability gate Not passed: current round skipped head search. Inspect saved raw outputs and improve the protocol in a new development run. ")
atomic_json(RUN_DIR / "pilot_writer_gate.json", WRITER_GATE)

RuntimeError: Live definitions/constants/config differ from the run manifest. Start a new run rather than mixing checkpoints.

## 07 ·  Training  H_R， Develop selection  K

 Keep the old book.  Gumbel straight-through mask  Search. Loss is  `-attack_margin + clean_weight × clean_harm + sparsity × number_of_dropped_heads`。train  Optimize. dev  Selection  K，test  _Other Organiser  heads  Its not the only way to evaluate a complete act.  margin  Declare success. 

In [ ]:
READ_SEARCHES, READ_SELECTIONS = [], []
READ_SELECTION = None
H_R, READ_CONTROLS, DEV_READ = [], {}, []
if RUN_DEVELOPMENT and CAPABILITY["eligible"]:
    train_attack, train_clean, train_excluded = snapshot_pairs(backend, DATA["train"], "actor")
    dev_attack, dev_clean, dev_excluded = snapshot_pairs(backend, DATA["dev"], "actor")
    atomic_json(RUN_DIR / "search" / "read-token-exclusions.json", {"train":train_excluded,"dev":dev_excluded})
    for search_seed in CFG["search_seeds"]:
        ranking = train_mask(backend, train_attack, train_clean, CFG, "read", search_seed)
        selection = select_heads_dev(backend, ranking, dev_attack, dev_clean, CFG, "read")
        READ_SEARCHES.append(ranking); READ_SELECTIONS.append(selection)
    READ_SELECTION = READ_SELECTIONS[0]  # predeclared primary seed, never best-of-seeds
    H_R = READ_SELECTION["heads"]
    display(pd.DataFrame(READ_SELECTION["candidates"]).drop(columns=["heads"]))
    if H_R:
        READ_CONTROLS = {f"random_{seed}": same_layer_random_heads(H_R,N_LAYERS,N_HEADS,seed)
                         for seed in CFG["random_seeds"]}
    else:
        print(" No development set to be met  H_R； Keep all the candidates and not force positive results. ")
    print("Primary read heads:", H_R)

## 08 ·  complete  Actor  Behaviour and comparison 

 Six paired conditions: correct facts only, another customer fact, approximately length-matched  metadata、 replacing distractor values, adding useful facts, and authorized updates. Do not restrict the primary analysis to originally successful attacks. Multiple  search seeds  Separate output, not as an additional stand-alone sample. 

In [ ]:
DEV_READ_EFFECTS = []
if RUN_DEVELOPMENT and CAPABILITY["eligible"]:
    read_head_sets = {"none": []}
    if H_R:
        read_head_sets.update({"H_R": H_R, **READ_CONTROLS})
    for selection in READ_SELECTIONS[1:]:
        if selection["eligible"]:
            read_head_sets[f"H_R_seed{selection['seed']}"] = selection["heads"]
    DEV_READ = evaluate_read(backend, DATA["dev"], read_head_sets, CFG)
    display(pd.DataFrame(export_rows("dev-read", DEV_READ, ["condition", "intervention"],
                                     ["task_success", "distractor_action", "format_failure", "stock_check_adherence", "snapshot_margin"])))
    if H_R:
        DEV_READ_EFFECTS = [dict(condition=condition, **paired_effect(DEV_READ, "intervention", "none", "H_R",
                             filters={"condition":condition}, n_boot=CFG["bootstrap_reps"])) for condition in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "dev-read-paired-effects.json", DEV_READ_EFFECTS)
        display(pd.DataFrame(DEV_READ_EFFECTS))

## 09 ·  Freezing experience, training  H_W

 Just...  untreated Actor  Make a recording of actual experience. All.  Writer  interventions read the same records; do not treat  Actor  The different experiences of the intervention are mixed. / development remains separate. Stopping the writer branch is a feasibility result, not a reason to fabricate replacement natural errors. 

In [ ]:
WRITE_SEARCHES, WRITE_SELECTIONS = [], []
WRITE_SELECTION = None
H_W, WRITE_CONTROLS, DEV_WRITER = [], {}, []
TRAIN_HISTORIES, DEV_HISTORIES = {}, {}
if RUN_DEVELOPMENT and CAPABILITY["eligible"] and WRITER_GATE and WRITER_GATE["eligible"]:
    TRAIN_HISTORIES = collect_fixed_histories(backend, DATA["train"], ("distractor", "correct_only"), CFG)
    DEV_HISTORIES = collect_fixed_histories(backend, DATA["dev"], CONDITIONS, CFG)
    train_attack, train_clean, train_excluded = snapshot_pairs(backend, DATA["train"], "writer", TRAIN_HISTORIES)
    dev_attack, dev_clean, dev_excluded = snapshot_pairs(backend, DATA["dev"], "writer", DEV_HISTORIES)
    atomic_json(RUN_DIR / "search" / "write-token-exclusions.json", {"train":train_excluded,"dev":dev_excluded})
    for search_seed in CFG["search_seeds"]:
        ranking = train_mask(backend, train_attack, train_clean, CFG, "write", search_seed)
        selection = select_heads_dev(backend, ranking, dev_attack, dev_clean, CFG, "write")
        WRITE_SEARCHES.append(ranking); WRITE_SELECTIONS.append(selection)
    WRITE_SELECTION = WRITE_SELECTIONS[0]
    H_W = WRITE_SELECTION["heads"]
    if H_W:
        WRITE_CONTROLS = {f"random_{seed}": same_layer_random_heads(H_W,N_LAYERS,N_HEADS,seed)
                          for seed in CFG["random_seeds"]}
    display(pd.DataFrame(WRITE_SELECTION["candidates"]).drop(columns=["heads"]))
    print("Primary write heads:", H_W)
else:
    print(" Side search not enabled in this round: check Actor Capacity, Writer Natural error gate and RUN_DEVELOPMENT。 The reading side results are still valid. ")

## 10 ·  complete  Writer  Evaluation 

 Targets incorrectly tied, correctly retained, omitted, duplicated, source mismatched and  JSON  Failures are reported separately; “delete the facts of the target” cannot be considered as a correcting error. H_R  in  Writer  Go, go, go! H_W  in  Actor  The cross-application is just a role-specific diagnosis. Its not supposed to be.  shared。

In [ ]:
if DEV_HISTORIES:
    write_head_sets = {"none": []}
    if H_W:
        write_head_sets.update({"H_W": H_W, **WRITE_CONTROLS})
    if H_R:
        write_head_sets["H_R_cross_role"] = H_R
    for selection in WRITE_SELECTIONS[1:]:
        if selection["eligible"]:
            write_head_sets[f"H_W_seed{selection['seed']}"] = selection["heads"]
    DEV_WRITER = evaluate_writer(backend, DATA["dev"], DEV_HISTORIES, write_head_sets, CFG)
    display(pd.DataFrame(export_rows("dev-writer", DEV_WRITER, ["condition","intervention"],
                                     ["target_distractor_error", "target_correct", "omission", "duplicate_count", "source_mismatch_count", "format_failure"])))
    if H_W:
        writer_effects = [dict(condition=condition, **paired_effect(DEV_WRITER, "intervention", "none", "H_W",
                           metric="target_correct", filters={"condition":condition}, n_boot=CFG["bootstrap_reps"])) for condition in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "dev-writer-paired-effects.json", writer_effects)
        CROSS_READ = evaluate_read(backend, DATA["dev"], {"H_W_cross_role":H_W}, CFG, conditions=("correct_only","distractor"))
        export_rows("dev-cross-read", CROSS_READ, ["condition","intervention"], ["task_success","format_failure"])

## 11 ·  Naturally.  memory   2×2  Follow-up tasks 

 Read only. / Write two sets.  heads  Completely run by development selection  2×2， Avoid drawing empty intervention into an independent processing group. Main analysis uses all  source  Nature.  M0/M1， Includes failure of format; no case of  error fixed screened. 

Y00 =  Original  Writer /  Original  Actor；Y10 = H_W /  Original  Actor；Y01 =  Original  Writer / H_R；Y11 = H_W / H_R。 Overall  memory  The dissemination effect of differences cannot alone prove the mechanism of a given field. 

In [ ]:
DEV_PIPELINE = None
if H_R and H_W and DEV_HISTORIES:
    DEV_PIPELINE = evaluate_memory_pipeline(backend, DATA["dev"], DEV_HISTORIES, H_R, H_W, CFG)
    atomic_json(RUN_DIR / "results" / "dev-memory-pipeline.json", DEV_PIPELINE)
    display(pd.DataFrame(export_rows("dev-factorial", DEV_PIPELINE["factorial"], ["arm"],
                                     ["task_success","distractor_action","format_failure"])))
    effects = factorial_contrasts(DEV_PIPELINE["factorial"])
    atomic_json(RUN_DIR / "results" / "dev-factorial-contrasts.json", effects)
    display(pd.DataFrame(effects))
else:
    print(" complete 2×2 Not running: Non-empty required H_R、H_W The study is based on the results already observed. ")

## 12 ·  Local  value  Replacement diagnostics 

 Limited  M0  The natural target was wrongly tied. M1  Correctly and with a single target entry  source； Number of applications clearly demonstrated / total. Partially replaced is a synonym field, not an entire one.  memory  String just happens to change only one  tokenizer token； The main experiment.  single-token  The statement is only for certified prefixes. 

In [ ]:
if DEV_PIPELINE:
    local = DEV_PIPELINE["local"]
    eligibility = [{"source_id":r["source_id"],"eligible":r["eligible"],"reason":r["reason"]} for r in local]
    local_rows = [row for result in local for row in result["rows"]]
    _atomic_csv(RUN_DIR / "results" / "dev-local-eligibility.csv", eligibility)
    print("Local eligible:", sum(r["eligible"] for r in local), "/", len(local))
    if local_rows:
        display(pd.DataFrame(export_rows("dev-local-swaps",local_rows,["variant"],["task_success"])))
        local_effects = [paired_effect(local_rows,"variant",a,b,n_boot=CFG["bootstrap_reps"])
                         for a,b in (("M0","M0_corrected"),("M1","M1_corrupted"))]
        atomic_json(RUN_DIR / "results" / "dev-local-effects.json", local_effects)
        display(pd.DataFrame(local_effects))

## 13 ·  Optional online update 

 Thats one step that goes on during the follow-up mission.  memory。Writer  Received Only Existing  memory  ♪ And real new moves ♪ / Feedback does not re-infuse clean truth values; true values are used only for rating. horizon 1/3/5  Its the same.  source  The observations in question cannot be merged into a stand-alone sample. 

In [ ]:
ONLINE_RESULTS = []
if RUN_ONLINE and DEV_PIPELINE:
    for case in DATA["dev"]:
        pair = DEV_PIPELINE["pairs"][case["source_id"]]
        for arm, memory_label, read_heads, write_heads in (("none","M0",[],[]),("both","M1",H_R,H_W)):
            for horizon in CFG["online_horizons"]:
                payload = {"case":case,"memory":pair[memory_label]["memory"],"arm":arm,
                           "read_heads":read_heads,"write_heads":write_heads,"horizon":horizon,"cfg":CFG}
                result = checkpoint_item("online", f"{case['source_id']}-{arm}-{horizon}",payload,
                    lambda: run_online(backend,case,pair[memory_label]["memory"],read_heads,write_heads,horizon,CFG))
                ONLINE_RESULTS.append({"source_id":case["source_id"],"arm":arm,"horizon":horizon,**result})
    atomic_json(RUN_DIR / "results" / "dev-online-exploratory.json", ONLINE_RESULTS)
    print("Online trajectories saved:",len(ONLINE_RESULTS))

## 14 ·  Freezing of protocol and then one-off assessment  test

 Default close. Completed.  FULL  After development, execute separately  `FREEZE_AND_RUN_TEST = True`， And run down below.  test cell； Not here.  QUICK  Upstart formal  test。 No need to rerun configuration  cell  Or a search ahead. 

 Agreement fixed model  revision、 Dependence, core definition / Mission Constant  hash、 Data disaggregation, master  seed、heads、 Random contrasts, main endpoints. test  Recreate your own experiences and  M0/M1， No, I do not.  dev memory。 View  test  Were not going to change it.  K、prompt  or sifting samples; reusable  checkpoint  Complete the same test that was interrupted. 

In [ ]:
TEST_READ, TEST_WRITER, TEST_PIPELINE = [], [], None
if FREEZE_AND_RUN_TEST:
    if not CAPABILITY["eligible"] or READ_SELECTION is None:
        raise RuntimeError("Finish the capability pilot and development stages first.")
    if not H_R and not H_W:
        raise RuntimeError("No eligible intervention selected; report the feasibility result instead of creating a positive test arm.")
    FROZEN = freeze_protocol(READ_SELECTION,WRITE_SELECTION,READ_CONTROLS,WRITE_CONTROLS,DATA_HASHES,WRITER_GATE)
    verify_protocol(FROZEN, DATA)
    test_read_sets = {"none":[]}
    if FROZEN["read_heads"]:
        test_read_sets.update({"H_R":FROZEN["read_heads"],**FROZEN["read_controls"]})
    TEST_READ = evaluate_read(backend,DATA["test"],test_read_sets,CFG)
    display(pd.DataFrame(export_rows("test-read",TEST_READ,["condition","intervention"],
                                     ["task_success","distractor_action","format_failure","stock_check_adherence"])))
    if FROZEN["read_heads"]:
        test_effects = [dict(condition=c,**paired_effect(TEST_READ,"intervention","none","H_R",
                            filters={"condition":c},n_boot=CFG["bootstrap_reps"])) for c in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "test-read-effects.json",test_effects)
    if FROZEN["writer_branch"]:
        TEST_HISTORIES = collect_fixed_histories(backend,DATA["test"],CONDITIONS,CFG)
        TEST_WRITER = evaluate_writer(backend,DATA["test"],TEST_HISTORIES,
                                      {"none":[],"H_W":FROZEN["write_heads"],**FROZEN["write_controls"]},CFG)
        export_rows("test-writer",TEST_WRITER,["condition","intervention"],
                    ["target_distractor_error","target_correct","omission","format_failure"])
        test_writer_effects = [dict(condition=c,**paired_effect(TEST_WRITER,"intervention","none","H_W",
                                   metric="target_correct",filters={"condition":c},n_boot=CFG["bootstrap_reps"])) for c in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "test-writer-effects.json",test_writer_effects)
        if FROZEN["read_heads"]:
            TEST_PIPELINE = evaluate_memory_pipeline(backend,DATA["test"],TEST_HISTORIES,FROZEN["read_heads"],FROZEN["write_heads"],CFG)
            atomic_json(RUN_DIR / "results" / "test-memory-pipeline.json",TEST_PIPELINE)
            export_rows("test-factorial",TEST_PIPELINE["factorial"],["arm"],["task_success","format_failure"])
            atomic_json(RUN_DIR / "results" / "test-factorial-effects.json",factorial_contrasts(TEST_PIPELINE["factorial"]))
    atomic_json(RUN_DIR / "test_complete.json",{"protocol_hash":stable_hash(FROZEN),"test_sources":len(DATA["test"]),"completed":True})
    print("Frozen test complete. Results:",RUN_DIR / "results")
else:
    print(" confirmation test closing; current round output only pilot/dev。")

## 15 ·  Save and read results 

- `manifest.json` / `model_revision.json` / `data_manifest.json`： This configuration, real model version and  source  Division. 
- `items/`： Case by case  raw generation、 Tool feedback, memory  and the cause of failure; resumes at the original configuration when the line is broken. 
- `search/`：mask logits、 Full Optimizer /RNG checkpoint、 each  seed  and  dev  Choose. 
- `results/*-summary.csv`： Press  source  aggregated tables; `*-effects.json`： Matching differences and compartments. 
- `frozen_protocol.json` / `test_complete.json`： Only officially.  test  Arising from implementation. 

 Read first.  clean  Capabilities and Formatting, Read Again  distractor  Behavior and Random  heads  contrasts;rejudgment  Writer  Is there a natural fact that was wrongly tied?  memory  whether downstream tasks are affected. Fewer errors accompanied by more omissions or impaired authorized updates are not a cost-free repair. 

 Keep the master model as the old version  base 8B。 If it does not fit into the complete tool, /Writer  Agreement, where available, to locate this limitation;  Instruct  or the change hint should be run independently and cannot be mixed with the current comparison. 

 The interface is based on: [TransformerLens 2.17  Model Interface ](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/transformer_lens/HookedTransformer.py)、[KV cache](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/transformer_lens/past_key_value_caching.py)、[ Declaration of reliance ](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/pyproject.toml)。

In [ ]:
RUN_STATUS = {"quick_mode":QUICK_MODE,"capability":CAPABILITY,"writer_gate":WRITER_GATE,
              "read_heads":H_R,"write_heads":H_W,"dev_read_rows":len(DEV_READ),
              "dev_writer_rows":len(DEV_WRITER),"dev_factorial_rows":len(DEV_PIPELINE["factorial"]) if DEV_PIPELINE else 0,
              "test_executed":bool(TEST_READ),"output_dir":str(RUN_DIR)}
atomic_json(RUN_DIR / "run_status.json",RUN_STATUS)
display(RUN_STATUS)
print(" Give priority to sharing when continuing analysis  run_status.json、summary.csv  and related  raw  Cases of failure. ")